In [1]:
# Импорт стандартных библиотек
import os
import warnings

# Импорт библиотек для анализа данных и визуализации
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт библиотек для машинного обучения
import joblib
from sklearn.base import BaseEstimator, RegressorMixin, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    fbeta_score,
    precision_score,
    recall_score,
    accuracy_score
)

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Предупреждения
warnings.filterwarnings('ignore')

# Визуальные стили
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Создание датасетов для машинного обучения

In [3]:
df = pd.read_csv('data.csv')
df.head()

,bp_number,bp_name,bp_is_active,bp_status,risk_name,risk_fin_loss,risk_business_rep,risk_rating,present_risk_zone,residual_risk_zone,event_name,event_date,risk_event_source_group,risk_event_reason_group,consequence_name,consequence_type,consequence_kind,loss_sum,reimburse_sum,consequence_influence_level,year_month
0,1,Описание бизнес-процесса 0001,1,ACTUAL,РC-0001,0,2,3,YELLOW,YELLOW,РСК-0001,2023-11-23 15:53:01,2,NO_REASON,РПС-0001,ACTIVITY,FACT,0.0,0.0,1,2023-11
1,1,Описание бизнес-процесса 0001,1,ACTUAL,РC-0001,0,2,3,YELLOW,YELLOW,РСК-0001,2023-11-23 15:53:01,1,NO_REASON,РПС-0001,ACTIVITY,FACT,0.0,0.0,1,2023-11
2,1,Описание бизнес-процесса 0001,1,ACTUAL,РC-0001,0,2,3,YELLOW,YELLOW,РСК-0001,2023-11-23 15:53:01,1,4,РПС-0001,ACTIVITY,FACT,0.0,0.0,1,2023-11
3,1,Описание бизнес-процесса 0001,1,ACTUAL,РC-0001,0,2,3,YELLOW,YELLOW,РСК-0002,2023-11-16 15:19:05,2,NO_REASON,РПС-0002,ACTIVITY,FACT,0.0,0.0,1,2023-11
4,1,Описание бизнес-процесса 0001,1,ACTUAL,РC-0001,0,2,3,YELLOW,YELLOW,РСК-0002,2023-11-16 15:19:05,1,NO_REASON,РПС-0002,ACTIVITY,FACT,0.0,0.0,1,2023-11


In [4]:
def create_combined_datasets(df, top_bp=20, re_count=100):
    """
    Создаёт оба датасета из одного источника, устраняя дублирование операций.
    
    Параметры:
    ----------
    df : pandas.DataFrame
        Исходный датасет с риск-событиями
    top_bp : int
        Количество топ бизнес-процессов для включения 
    re_count : int
        Минимальное количество встречаемости рисков
    
    Возвращает:
    -----------
    tuple
        (dataset1, dataset2) – оба датасета с сохранением оригинальной логики
    """
    
    ############################################################################
    # ЭТАП 1: ПРЕДВАРИТЕЛЬНАЯ ФИЛЬТРАЦИЯ 
    ############################################################################
    print("=" * 80)
    print("ПРЕДВАРИТЕЛЬНАЯ ФИЛЬТРАЦИЯ ")
    print("=" * 80)
    
    # Приведение столбца event_date к datetime
    if not pd.api.types.is_datetime64_any_dtype(df['event_date']):
        df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
    
    print(f"Исходный диапазон дат: {df['event_date'].min()} - {df['event_date'].max()}")
    
    # 1. Ограничение данных периодом 2020–2026 гг.
    start_date = pd.Timestamp('2020-01-01')
    end_date = pd.Timestamp('2026-12-31')
    
    df_filtered_time = df[(df['event_date'] >= start_date) & (df['event_date'] <= end_date)].copy()
    print(f"Данные после фильтрации по периоду 2020-2026: {len(df_filtered_time)} записей")
    print(f"Диапазон дат после фильтрации: {df_filtered_time['event_date'].min()} - {df_filtered_time['event_date'].max()}")
    
    # 2. Отбор бизнес-процессов: минимум 100 событий всего и 50 за последний год
    print("\nФильтрация бизнес-процессов по условиям...")
    
    last_date = df_filtered_time['event_date'].max()
    last_year_start = pd.Timestamp(last_date.year, 1, 1)
    
    bp_stats = df_filtered_time.groupby('bp_number').agg(
        total_events=('event_date', 'count'),
        events_last_year=('event_date', lambda x: ((x >= last_year_start) & (x <= last_date)).sum())
    ).reset_index()
    
    filtered_bp = bp_stats[
        (bp_stats['total_events'] > 100) & 
        (bp_stats['events_last_year'] > 50)
    ]['bp_number'].tolist()
    
    print(f"Бизнес-процессов с более чем 100 событий в общем: {(bp_stats['total_events'] > 100).sum()}")
    print(f"Бизнес-процессов с более чем 50 событий в последнем году: {(bp_stats['events_last_year'] > 50).sum()}")
    print(f"Бизнес-процессов, удовлетворяющих обоим условиям: {len(filtered_bp)}")
    
    if len(filtered_bp) == 0:
        print("ВНИМАНИЕ: Ни один бизнес-процесс не удовлетворяет условиям!")
        print("Используем все доступные бизнес-процессы")
        filtered_bp = df_filtered_time['bp_number'].unique().tolist()
    else:
        # Ограничение топ-N бизнес-процессов по общему количеству событий
        if len(filtered_bp) > top_bp:
            print(f"Ограничиваем до топ-{top_bp} бизнес-процессов по общему количеству событий")
            top_bp_stats = bp_stats[bp_stats['bp_number'].isin(filtered_bp)].sort_values('total_events', ascending=False).head(top_bp)
            filtered_bp = top_bp_stats['bp_number'].tolist()
    
    top_bp_list = filtered_bp
    
    print(f"\nОтобранные бизнес-процессы ({len(top_bp_list)}):")
    for i, bp in enumerate(top_bp_list, 1):
        total = bp_stats[bp_stats['bp_number'] == bp]['total_events'].values[0]
        last_year = bp_stats[bp_stats['bp_number'] == bp]['events_last_year'].values[0]
        print(f"  {i}. {bp}: всего {total} событий, в последнем году {last_year}")
    
    df_top = df_filtered_time[df_filtered_time['bp_number'].isin(top_bp_list)]
    
    # 3. Отбор рисков, встречающихся не менее re_count раз
    risk_counts = df_top['risk_name'].value_counts()
    valid_risks = risk_counts[risk_counts >= re_count].index.tolist()
    print(f"\nРисков, встречающихся не менее {re_count} раз: {len(valid_risks)} из {len(risk_counts)}")
    
    df_filtered = df_top[df_top['risk_name'].isin(valid_risks)].copy()
    
    # 4. Добавление колонки год-месяц для временной агрегации
    df_filtered['year_month'] = df_filtered['event_date'].dt.to_period('M').astype(str)
    
    print(f"\nФИНАЛЬНЫЙ ОТФИЛЬТРОВАННЫЙ ДАТАСЕТ:")
    print(f"  Количество записей: {len(df_filtered)}")
    print(f"  Уникальных бизнес-процессов: {df_filtered['bp_number'].nunique()}")
    print(f"  Уникальных рисков: {df_filtered['risk_name'].nunique()}")
    print(f"  Диапазон дат: {df_filtered['event_date'].min()} - {df_filtered['event_date'].max()}")
    
    ############################################################################
    # ЭТАП 2: СОЗДАНИЕ КОПИЙ ДЛЯ КАЖДОГО ДАТАСЕТА
    ############################################################################
    print("\n" + "=" * 80)
    print("СОЗДАНИЕ ОБОИХ ДАТАСЕТОВ ИЗ ОТФИЛЬТРОВАННЫХ ДАННЫХ")
    print("=" * 80)
    
    df_common = df_filtered.copy()      # Общие данные
    df1 = df_common.copy()               # Для датасета 1
    df2 = df_common.copy()               # Для датасета 2
    
   ############################################################################
    # ДАТАСЕТ 1: ВЕРОЯТНОСТЬ СРАБАТЫВАНИЯ ИСТОЧНИКА РИСКА
    ############################################################################
    print("\n" + "=" * 80)
    print("СОЗДАНИЕ ДАТАСЕТА 1 (вероятность срабатывания источника риска)")
    print("=" * 80)
    
    def create_dataset1(df1_input):
        """Оригинальная логика Кода 1 в виде функции"""
        df = df1_input.copy()
              
        # 2. Фильтрация рисков: только риски с более чем 100 риск-событий
        risk_event_counts = df.groupby(['bp_number', 'risk_name']).size()
        risks_with_many_events = risk_event_counts[risk_event_counts > 100].index.tolist()
        df = df[df.set_index(['bp_number', 'risk_name']).index.isin(risks_with_many_events)].copy()
        
        # 3. Создание временной метки year_month
        df['year_month'] = df['event_date'].dt.to_period('M').astype(str)
        
        # 4. Кодирование категориальных переменных
        zone_mapping = {'GREEN': 0, 'YELLOW': 1, 'RED': 2}
        df['present_risk_zone_encoded'] = df['present_risk_zone'].map(zone_mapping).fillna(-1).astype(int)
        df['residual_risk_zone_encoded'] = df['residual_risk_zone'].map(zone_mapping).fillna(-1).astype(int)
        
        categorical_cols = [
            'risk_rating', 'risk_fin_loss', 'risk_business_rep',
            'consequence_type', 'consequence_influence_level'
        ]
        
        encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        
        # 5. (Удалён шаг замены 'NO_SOURCE' на '0' – оставляем исходные значения)
        
        # 6. Агрегация по месяцам с дополнительными колонками для каждого источника
        #    Приводим колонку risk_event_source_group к строке, чтобы корректно сравнивать со строковыми идентификаторами источников
        df['risk_event_source_group'] = df['risk_event_source_group'].astype(str)
        
        for source in ['1', '2', '3', '4']:
            df[f'source_{source}_count'] = (df['risk_event_source_group'] == source).astype(int)
        
        # Проверка, что счётчики не нулевые (для отладки)
        print("Проверка сумм счётчиков источников после создания:")
        print(df[['source_1_count', 'source_2_count', 'source_3_count', 'source_4_count']].sum())
        
        agg_dict = {
            'risk_rating_encoded': 'first',
            'risk_fin_loss_encoded': 'first',
            'risk_business_rep_encoded': 'first',
            'present_risk_zone_encoded': 'first',
            'residual_risk_zone_encoded': 'first',
            'consequence_type_encoded': 'first',
            'consequence_influence_level_encoded': 'first',
            'loss_sum': 'sum',
            'reimburse_sum': 'sum',
            'source_1_count': 'sum',
            'source_2_count': 'sum',
            'source_3_count': 'sum',
            'source_4_count': 'sum'
        }
        
        monthly_data = df.groupby(['bp_number', 'risk_name', 'year_month']).agg(agg_dict).reset_index()
        
        # 7. Добавление количества событий
        event_counts = df.groupby(['bp_number', 'risk_name', 'year_month']).size().reset_index(name='event_count')
        monthly_data = monthly_data.merge(event_counts, on=['bp_number', 'risk_name', 'year_month'])
        
        # 8. Переименование колонок
        monthly_data = monthly_data.rename(columns={
            'loss_sum': 'total_loss',
            'reimburse_sum': 'total_reimburse'
        })
        
        # 9. СОРТИРОВКА по дате
        monthly_data['date_dt'] = pd.to_datetime(monthly_data['year_month'] + '-01')
        monthly_data = monthly_data.sort_values(['bp_number', 'risk_name', 'date_dt']).reset_index(drop=True)
        
        # 10. Создание полного набора дат для каждого риска
        date_range = pd.date_range('2020-01-01', '2025-11-01', freq='MS')
        all_months = [d.strftime('%Y-%m') for d in date_range]
        
        result_rows = []
        
        for (bp_num, risk_name), group in monthly_data.groupby(['bp_number', 'risk_name']):
            full_dates = pd.DataFrame({
                'bp_number': bp_num,
                'risk_name': risk_name,
                'year_month': all_months,
                'date_dt': pd.to_datetime([m + '-01' for m in all_months])
            })
            
            group_full = pd.merge(full_dates, group,
                                 on=['bp_number', 'risk_name', 'year_month', 'date_dt'],
                                 how='left')
            
            static_cols = ['risk_rating_encoded', 'risk_fin_loss_encoded', 'risk_business_rep_encoded',
                           'present_risk_zone_encoded', 'residual_risk_zone_encoded',
                           'consequence_type_encoded', 'consequence_influence_level_encoded']
            
            for col in static_cols:
                if group_full[col].isna().any() and not group_full[col].isna().all():
                    group_full[col] = group_full[col].fillna(method='ffill').fillna(method='bfill')
            
            dynamic_cols = ['event_count', 'total_loss', 'total_reimburse',
                           'source_1_count', 'source_2_count', 'source_3_count', 'source_4_count']
            
            for col in dynamic_cols:
                group_full[col] = group_full[col].fillna(0)
            
            group_full = group_full.sort_values('date_dt').reset_index(drop=True)
            
            for i in range(len(group_full)):
                row = group_full.iloc[i].copy()
                
                for lag in [1, 2, 3]:
                    if i >= lag:
                        row[f'event_count_lag_{lag}'] = group_full.iloc[i - lag]['event_count']
                    else:
                        row[f'event_count_lag_{lag}'] = np.nan
                
                if i > 0:
                    prev_events = group_full.iloc[max(0, i-3):i]['event_count']
                    row['event_count_rolling_mean_3'] = prev_events.mean() if len(prev_events) > 0 else np.nan
                    row['event_count_rolling_std_3'] = prev_events.std() if len(prev_events) > 1 else np.nan
                    
                    prev_events_6 = group_full.iloc[max(0, i-6):i]['event_count']
                    row['event_count_rolling_mean_6'] = prev_events_6.mean() if len(prev_events_6) > 0 else np.nan
                    row['event_count_rolling_std_6'] = prev_events_6.std() if len(prev_events_6) > 1 else np.nan
                    
                    for source in ['1', '2', '3', '4']:
                        if i >= 1:
                            row[f'source_{source}_count_lag_1'] = group_full.iloc[i-1][f'source_{source}_count']
                        else:
                            row[f'source_{source}_count_lag_1'] = np.nan
                else:
                    row['event_count_rolling_mean_3'] = np.nan
                    row['event_count_rolling_std_3'] = np.nan
                    row['event_count_rolling_mean_6'] = np.nan
                    row['event_count_rolling_std_6'] = np.nan
                    for source in ['1', '2', '3', '4']:
                        row[f'source_{source}_count_lag_1'] = np.nan
                
                if i > 0:
                    months_since = 0
                    for j in range(i-1, -1, -1):
                        if group_full.iloc[j]['event_count'] > 0:
                            months_since = i - j
                            break
                    row['months_since_last_event'] = months_since
                else:
                    row['months_since_last_event'] = 0
                
                row['month_of_year'] = row['date_dt'].month
                
                if i < len(group_full) - 1:
                    for source in ['1', '2', '3', '4']:
                        next_source_count = group_full.iloc[i+1][f'source_{source}_count']
                        row[f'target_source_{source}'] = 1 if next_source_count > 0 else 0
                else:
                    for source in ['1', '2', '3', '4']:
                        row[f'target_source_{source}'] = np.nan
                
                result_rows.append(row)
        
        final_dataset = pd.DataFrame(result_rows)
        
        # 12. Создание дополнительных признаков для каждого источника
        for source in ['1', '2', '3', '4']:
            final_dataset[f'source_{source}_ratio'] = final_dataset[f'source_{source}_count'] / final_dataset['event_count']
            final_dataset[f'source_{source}_ratio'] = final_dataset[f'source_{source}_ratio'].fillna(0)
        
        # 13. Выбор финальных колонок
        final_columns = [
            'bp_number', 'risk_name', 'year_month',
            'risk_rating_encoded', 'risk_fin_loss_encoded', 'risk_business_rep_encoded',
            'present_risk_zone_encoded', 'residual_risk_zone_encoded',
            'consequence_type_encoded', 'consequence_influence_level_encoded',
            'event_count', 'total_loss', 'total_reimburse',
            'source_1_count', 'source_2_count', 'source_3_count', 'source_4_count',
            'source_1_ratio', 'source_2_ratio', 'source_3_ratio', 'source_4_ratio',
            'event_count_lag_1', 'event_count_lag_2', 'event_count_lag_3',
            'event_count_rolling_mean_3', 'event_count_rolling_std_3',
            'event_count_rolling_mean_6', 'event_count_rolling_std_6',
            'months_since_last_event', 'month_of_year'
        ]
        
        for source in ['1', '2', '3', '4']:
            final_columns.append(f'source_{source}_count_lag_1')
        
        final_columns.extend(['target_source_1', 'target_source_2', 'target_source_3', 'target_source_4'])
        
        final_dataset = final_dataset[final_columns].reset_index(drop=True)
        
        # 14. Финальная сортировка по дате
        final_dataset['date_dt'] = pd.to_datetime(final_dataset['year_month'] + '-01')
        final_dataset = final_dataset.sort_values(['bp_number', 'risk_name', 'date_dt']).reset_index(drop=True)
        
        # 15. Заполнение пропусков в признаках
        for lag in [1, 2, 3]:
            final_dataset[f'event_count_lag_{lag}'] = final_dataset[f'event_count_lag_{lag}'].fillna(0)
        
        for source in ['1', '2', '3', '4']:
            lag_col = f'source_{source}_count_lag_1'
            if lag_col in final_dataset.columns:
                final_dataset[lag_col] = final_dataset[lag_col].fillna(0)
        
        for col in ['event_count_rolling_mean_3', 'event_count_rolling_std_3',
                    'event_count_rolling_mean_6', 'event_count_rolling_std_6']:
            mean_value = final_dataset[col].mean()
            final_dataset[col] = final_dataset[col].fillna(mean_value)
        
        final_dataset['total_loss'] = final_dataset['total_loss'].fillna(0)
        final_dataset['total_reimburse'] = final_dataset['total_reimburse'].fillna(0)
        
        for source in ['1', '2', '3', '4']:
            count_col = f'source_{source}_count'
            if count_col in final_dataset.columns:
                final_dataset[count_col] = final_dataset[count_col].fillna(0)
        
        for source in ['1', '2', '3', '4']:
            ratio_col = f'source_{source}_ratio'
            final_dataset[ratio_col] = final_dataset[ratio_col].fillna(0)
        
        final_dataset = final_dataset.drop(columns=['date_dt'])
        
        # Анализ датасета
        print(f"\nФИНАЛЬНЫЙ ДАТАСЕТ:")
        print(f"Размер: {final_dataset.shape}")
        print(f"Количество признаков: {len(final_dataset.columns) - 4}")  # минус 4 целевых
        print(f"Период данных: от {final_dataset['year_month'].min()} до {final_dataset['year_month'].max()}")
        
        print("\n" + "="*60)
        print("РАСПРЕДЕЛЕНИЕ КЛАССОВ В ЦЕЛЕВЫХ ПЕРЕМЕННЫХ (ВСЕ ДАННЫЕ)")
        print("="*60)
        
        target_cols = ['target_source_1', 'target_source_2', 'target_source_3', 'target_source_4']
        for target in target_cols:
            valid_data = final_dataset[target].dropna()
            if len(valid_data) > 0:
                counts = valid_data.value_counts().sort_index()
                percentages = valid_data.value_counts(normalize=True).sort_index() * 100
                
                print(f"\n{target}:")
                print(f"  0 (нет событий): {counts.get(0, 0)} ({percentages.get(0, 0):.1f}%)")
                print(f"  1 (есть события): {counts.get(1, 0)} ({percentages.get(1, 0):.1f}%)")
                print(f"  Всего записей: {len(valid_data)}")
                print(f"  Пропусков (последний месяц): {final_dataset[target].isna().sum()}")
            else:
                print(f"\n{target}: Нет данных для анализа")
        
        print("\n" + "="*60)
        print("ИНФОРМАЦИЯ О ДАТАСЕТЕ")
        print("="*60)
        print(f"Записей: {len(final_dataset)}")
        print(f"Уникальных бизнес-процессов: {final_dataset['bp_number'].nunique()}")
        print(f"Уникальных рисков: {final_dataset['risk_name'].nunique()}")
        print(f"Месяцев на каждый риск: {len(final_dataset) // final_dataset['risk_name'].nunique() if final_dataset['risk_name'].nunique() > 0 else 0}")
        print(f"\nПример структуры данных:")
        final_dataset.tail()
        
        return final_dataset

    # ВЫЗОВ ФУНКЦИИ ДЛЯ ДАТАСЕТА 1
    dataset1 = create_dataset1(df1)
    
    ############################################################################
    # ДАТАСЕТ 2: ПРЕДСКАЗАНИЕ КОЛИЧЕСТВА РИСК-СОБЫТИЙ
    ############################################################################
    print("\n" + "=" * 80)
    print("СОЗДАНИЕ ДАТАСЕТА 2 (предсказание количества риск-событий)")
    print("=" * 80)
    
    def create_dataset2(df2_input, top_bp=20, re_count=100):
        """Оригинальная логика Кода 2 в виде функции"""
        df = df2_input.copy()
        
        # Вспомогательные функции для генерации признаков
        def add_lags_with_fill(df_group, col, lags):
            for lag in lags:
                lag_col = f'{col}_lag_{lag}'
                df_group[lag_col] = df_group[col].shift(lag).fillna(0)
            return df_group
        
        def add_rolling_stats(df_group, col, windows):
            for window in windows:
                df_group[f'{col}_rolling_mean_{window}'] = df_group[col].rolling(window=window, min_periods=1).mean()
                df_group[f'{col}_rolling_std_{window}'] = df_group[col].rolling(window=window, min_periods=1).std().fillna(0)
            return df_group
        
        def months_since_last_event(df_group):
            months_since = []
            last_event_month = -1
            for idx, row in enumerate(df_group.itertuples()):
                if row.event_count > 0:
                    last_event_month = idx
                    months_since.append(0)
                elif last_event_month >= 0:
                    months_since.append(idx - last_event_month)
                else:
                    months_since.append(999)
            return pd.Series(months_since, index=df_group.index)
        
        def remove_outliers_from_event_count(df, percentile=99):
            df_clean = df.copy()
            if 'event_count' in df_clean.columns:
                p_value = np.percentile(df_clean['event_count'], percentile)
                total_values = df_clean['event_count'].shape[0]
                outliers_count = (df_clean['event_count'] > p_value).sum()
                outliers_percentage = (outliers_count / total_values) * 100
                
                df_clean.loc[df_clean['event_count'] > p_value, 'event_count'] = p_value
                
                print(f"Удаление выбросов из колонки event_count ({percentile}-й перцентиль):")
                print(f"  - {percentile}-й перцентиль: {p_value:.2f}")
                print(f"  - Количество выбросов (> {p_value:.2f}): {outliers_count} записей")
                print(f"  - Процент выбросов: {outliers_percentage:.2f}%")
                print(f"  - Максимальное значение после обработки: {df_clean['event_count'].max():.2f}")
            else:
                print("Колонка 'event_count' не найдена в данных")
            return df_clean
        
        def print_business_process_summary(df, top_bp_list):
            print("\n" + "=" * 80)
            print("СВОДКА ПО БИЗНЕС-ПРОЦЕССАМ")
            print("=" * 80)
            
            for i, bp in enumerate(top_bp_list, 1):
                bp_data = df[df['bp_number'] == bp].copy()
                if bp_data.empty:
                    continue
                bp_data = bp_data.dropna(subset=['event_date'])
                if bp_data.empty:
                    continue
                earliest_date = bp_data['event_date'].min()
                latest_date = bp_data['event_date'].max()
                total_events = bp_data.shape[0]
                unique_risks = bp_data['risk_name'].nunique()
                
                print(f"{i}. Бизнес-процесс: {bp}")
                print(f"   - Самая ранняя дата: {earliest_date.strftime('%Y-%m-%d')}")
                print(f"   - Самая поздняя дата: {latest_date.strftime('%Y-%m-%d')}")
                print(f"   - Период охвата: {(latest_date - earliest_date).days} дней")
                print(f"   - Всего риск-событий: {total_events}")
                print(f"   - Уникальных типов рисков: {unique_risks}")
                
                top_risks = bp_data['risk_name'].value_counts().head(5)
                if not top_risks.empty:
                    print(f"   - Топ-5 рисков по частоте:")
                    for j, (risk_name, count) in enumerate(top_risks.items(), 1):
                        print(f"      {j}. {risk_name}: {count} событий")
                print()
        
        print("Шаг 1: Подготовка данных...")
        
        # Данные уже отфильтрованы, выводим статистику
        print(f"Диапазон дат: {df['event_date'].min()} - {df['event_date'].max()}")
        print(f"Количество записей: {len(df)}")
        
        print_business_process_summary(df, top_bp_list)
        
        print("\nШаг 2: Создание базовой структуры датасета...")
        
        # Извлечение уникальных атрибутов рисков (статические характеристики)
        risk_attributes = df[
            ['bp_number', 'risk_name', 
             'risk_fin_loss', 'risk_business_rep', 'risk_rating', 
             'present_risk_zone', 'residual_risk_zone',
             'risk_event_source_group', 'risk_event_reason_group',
             'consequence_name', 'consequence_type', 'consequence_kind',
             'consequence_influence_level', 'loss_sum', 'reimburse_sum']
        ].drop_duplicates(subset=['bp_number', 'risk_name'])
        
        duplicate_check = risk_attributes.groupby(['bp_number', 'risk_name']).size()
        if (duplicate_check > 1).any():
            print("Предупреждение: Найдены дублирующиеся атрибуты, используется первое значение")
            risk_attributes = risk_attributes.groupby(['bp_number', 'risk_name']).first().reset_index()
        
        # Формирование временной сетки (все месяцы от min до max)
        min_date = pd.Period(df['year_month'].min())
        max_date = pd.Period(df['year_month'].max())
        all_months = pd.period_range(start=min_date, end=max_date, freq='M')
        
        print(f"Период для временных рядов: {min_date} - {max_date}")
        print(f"Всего месяцев: {len(all_months)}")
        
        # Все комбинации (бизнес-процесс, риск, месяц)
        bp_risk_pairs = df[['bp_number', 'risk_name']].drop_duplicates()
        combinations = [
            {'bp_number': row['bp_number'], 'risk_name': row['risk_name'], 'year_month': month}
            for month in all_months
            for _, row in bp_risk_pairs.iterrows()
        ]
        
        base_df = pd.DataFrame(combinations)
        
        print("\nШаг 3: Добавление атрибутов и агрегация событий...")
        
        # Присоединение статических атрибутов
        base_df = base_df.merge(risk_attributes, on=['bp_number', 'risk_name'], how='left')
        
        # Агрегация фактических событий по месяцам
        event_counts = df.groupby(['bp_number', 'risk_name', 'year_month']).size().reset_index(name='event_count')
        event_counts['year_month'] = event_counts['year_month'].astype(str)
        base_df['year_month'] = base_df['year_month'].astype(str)
        base_df = base_df.merge(event_counts, on=['bp_number', 'risk_name', 'year_month'], how='left')
        base_df['event_count'] = base_df['event_count'].fillna(0)
        
        print("\nШаг 4: Удаление выбросов из event_count...")
        
        print("Статистика до удаления выбросов:")
        print(f"  - Среднее: {base_df['event_count'].mean():.2f}")
        print(f"  - Медиана: {base_df['event_count'].median():.2f}")
        print(f"  - Максимум: {base_df['event_count'].max():.2f}")
        print(f"  - 99-й перцентиль: {np.percentile(base_df['event_count'], 99):.2f}")
        
        base_df = remove_outliers_from_event_count(base_df, percentile=99)
        
        print("\nСтатистика после удаления выбросов:")
        print(f"  - Среднее: {base_df['event_count'].mean():.2f}")
        print(f"  - Медиана: {base_df['event_count'].median():.2f}")
        print(f"  - Максимум: {base_df['event_count'].max():.2f}")
        print(f"  - 99-й перцентиль: {np.percentile(base_df['event_count'], 99):.2f}")
        
        print("\nШаг 5: Кодирование признаков...")
        
        # Определение категориальных признаков с иерархией (порядковые)
        hierarchical_cats = {
            'risk_rating': {'hierarchy': True, 'order': [1, 2, 3, 4, 5]},
            'risk_fin_loss': {'hierarchy': True, 'order': [0, 1, 2, 3, 4, 5]},
            'present_risk_zone': {'hierarchy': True, 'order': ['GREEN', 'YELLOW', 'RED']},
            'residual_risk_zone': {'hierarchy': True, 'order': ['GREEN', 'YELLOW', 'RED']},
            'consequence_influence_level': {'hierarchy': True, 'order': [0, 1, 2, 3]},
            'risk_business_rep': {'hierarchy': True, 'order': [0, 1, 2, 3, 4]},
        }
        
        # Обычные категориальные признаки (без порядка)
        ordinary_cats = [
            'risk_event_source_group',
            'risk_event_reason_group', 
            'consequence_type',
            'consequence_kind',
            'consequence_name'
        ]
        
        # Числовые признаки (оставляем как есть)
        numeric_cats = ['loss_sum', 'reimburse_sum']
        
        # Кодирование порядковых признаков
        for col, info in hierarchical_cats.items():
            if info['hierarchy']:
                order_mapping = {value: idx for idx, value in enumerate(info['order'])}
                base_df[f'{col}_encoded'] = base_df[col].map(order_mapping).fillna(-1).astype(int)
        
        # Кодирование обычных категориальных признаков
        for col in ordinary_cats:
            base_df[col] = base_df[col].fillna('UNKNOWN')
            if col in ['risk_event_source_group', 'risk_event_reason_group']:
                # Эти колонки уже числовые, просто заполняем пропуски
                base_df[col] = pd.to_numeric(base_df[col], errors='coerce')
                base_df[col] = base_df[col].fillna(-1).astype(int)
                base_df[f'{col}_encoded'] = base_df[col]
            else:
                unique_vals = base_df[col].dropna().unique()
                label_mapping = {val: idx for idx, val in enumerate(unique_vals)}
                base_df[f'{col}_encoded'] = base_df[col].map(label_mapping).fillna(-1).astype(int)
        
        # Числовые признаки просто копируем
        for col in numeric_cats:
            base_df[col] = base_df[col].fillna(0).astype(float)
            base_df[f'{col}_encoded'] = base_df[col]
        
        print("\nШаг 6: Временные преобразования...")
        
        # Сортировка по бизнес-процессу, риску и месяцу
        base_df = base_df.sort_values(['bp_number', 'risk_name', 'year_month']).reset_index(drop=True)
        
        # Добавление лагов количества событий
        lags_to_add = [1, 2, 3, 4, 5, 6, 9, 12, 18, 24]
        base_df = base_df.groupby(['bp_number', 'risk_name']).apply(
            lambda x: add_lags_with_fill(x, 'event_count', lags_to_add)
        ).reset_index(drop=True)
        
        # Добавление скользящих статистик
        windows_to_add = [3, 6, 9, 12, 18, 24]
        base_df = base_df.groupby(['bp_number', 'risk_name']).apply(
            lambda x: add_rolling_stats(x, 'event_count', windows_to_add)
        ).reset_index(drop=True)
        
        print("\nШаг 7: Создание целевых переменных...")
        
        # Целевая переменная: факт события в следующем месяце
        base_df['target_next_month'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].shift(-1)
        base_df['target_next_month'] = (base_df['target_next_month'] > 0).astype(int)
        # Количество событий в следующем месяце (для регрессии)
        base_df['event_count_next_month'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].shift(-1)
        
        print("\nШаг 8: Преобразование дат и фильтрация...")
        
        base_df['year_month'] = base_df['year_month'].astype(object)
        base_df['year_month_dt'] = base_df['year_month'].apply(lambda x: pd.to_datetime(str(x) + '-01') if pd.notnull(x) else pd.NaT)
        
        # Повторная фильтрация по периоду (для удаления месяцев вне 2020-2026)
        start_filter_date = pd.Timestamp('2020-01-01')
        end_filter_date = pd.Timestamp('2026-12-31')
        base_df = base_df[(base_df['year_month_dt'] >= start_filter_date) & 
                          (base_df['year_month_dt'] <= end_filter_date)].copy()
        
        print(f"Данные после фильтрации по периоду 2020-2026: {len(base_df)} записей")
        print(f"Диапазон дат в датасете: {base_df['year_month_dt'].min()} - {base_df['year_month_dt'].max()}")
        
        # Удаление строк, где целевая переменная отсутствует (последний месяц)
        initial_count = len(base_df)
        base_df = base_df.dropna(subset=['target_next_month', 'event_count_next_month'])
        print(f"Удалено записей без целевой переменной: {initial_count - len(base_df)}")
        
        print("\nШаг 9: Добавление дополнительных признаков...")
        
        # Месяцев с последнего события
        base_df['months_since_last_event'] = base_df.groupby(['bp_number', 'risk_name']).apply(
            lambda x: months_since_last_event(x)
        ).reset_index(level=[0,1], drop=True)
        
        # Общее количество событий за весь период (для каждого риска)
        total_events = base_df.groupby(['bp_number', 'risk_name'])['event_count'].sum().reset_index(name='total_events')
        base_df = base_df.merge(total_events, on=['bp_number', 'risk_name'])
        
        print("\nШаг 10: Кодирование категориальных колонок...")
        
        # Бизнес-процесс и риск оставляем в исходном формате (не кодируем)
        print("Пропускаем кодирование колонок 'bp_number' и 'risk_name' для сохранения исходного формата")
        
        # Дополнительные временные и комбинированные признаки
        if 'year_month_dt' in base_df.columns and pd.api.types.is_datetime64_any_dtype(base_df['year_month_dt']):
            # Убедимся, что все нужные лаги присутствуют
            for lag in [1, 2, 3, 6, 9, 12, 18, 24]:
                col_name = f'event_count_lag_{lag}'
                if col_name not in base_df.columns:
                    base_df[f'event_count_lag_{lag}'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].shift(lag)
            
            for window in [3, 6, 9, 12, 18, 24]:
                mean_col = f'rolling_mean_{window}'
                std_col = f'rolling_std_{window}'
                if mean_col not in base_df.columns:
                    base_df[f'rolling_mean_{window}'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].transform(
                        lambda x: x.rolling(window, min_periods=1).mean())
                if std_col not in base_df.columns:
                    base_df[f'rolling_std_{window}'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].transform(
                        lambda x: x.rolling(window, min_periods=1).std())
            
            # Кумулятивные признаки
            base_df['months_since_first'] = base_df.groupby(['bp_number', 'risk_name']).cumcount()
            base_df['total_events_so_far'] = base_df.groupby(['bp_number', 'risk_name'])['event_count'].cumsum()
            
            # Циклические признаки месяца
            base_df['month'] = base_df['year_month_dt'].dt.month
            base_df['quarter'] = base_df['year_month_dt'].dt.quarter
            base_df['month_sin'] = np.sin(2 * np.pi * base_df['month'] / 12)
            base_df['month_cos'] = np.cos(2 * np.pi * base_df['month'] / 12)
            
            # Взаимодействие рейтинга с лагами
            if 'risk_rating_encoded' in base_df.columns:
                base_df['rating_x_lag1'] = base_df['risk_rating_encoded'] * base_df['event_count_lag_1'].fillna(0)
                base_df['rating_x_total'] = base_df['risk_rating_encoded'] * base_df['total_events_so_far']
        
        print("\nШаг 11: Финальная обработка...")
        
        # Заполнение оставшихся пропусков
        for col in base_df.select_dtypes(include=[np.number]).columns:
            if base_df[col].isnull().any():
                if 'lag' in col or 'rolling' in col:
                    base_df[col] = base_df[col].fillna(0)
                else:
                    base_df[col] = base_df[col].fillna(base_df[col].median())
        
        # Удаление исходных категориальных колонок, оставляем только закодированные
        original_cat_cols = list(hierarchical_cats.keys()) + ordinary_cats + numeric_cats
        cols_to_drop = original_cat_cols + ['year_month_dt']
        base_df = base_df.drop(cols_to_drop, axis=1, errors='ignore')
        
        # Гарантия отсутствия пропусков
        numeric_cols = base_df.select_dtypes(include=[np.number]).columns
        base_df[numeric_cols] = base_df[numeric_cols].fillna(0)
        base_df['event_count_next_month'] = base_df['event_count_next_month'].astype(int)
        
        return base_df
    
    dataset2 = create_dataset2(df2, top_bp=top_bp, re_count=re_count)
    
    ############################################################################
    # ВЫВОД ИНФОРМАЦИИ О РЕЗУЛЬТАТАХ
    ############################################################################
    print("\n" + "=" * 80)
    print("РЕЗУЛЬТАТЫ СОЗДАНИЯ ДАТАСЕТОВ")
    print("=" * 80)
    
    print(f"\nДАТАСЕТ 1 (вероятность срабатывания источника риска):")
    print(f"  Размер: {dataset1.shape}")
    print(f"  Период данных: от {dataset1['year_month'].min()} до {dataset1['year_month'].max()}")
    print(f"  Уникальных бизнес-процессов: {dataset1['bp_number'].nunique()}")
    print(f"  Уникальных рисков: {dataset1['risk_name'].nunique()}")
    
    print(f"\nДАТАСЕТ 2 (предсказание количества риск-событий):")
    print(f"  Размер: {dataset2.shape}")
    print(f"  Уникальных бизнес-процессов: {dataset2['bp_number'].nunique()}")
    print(f"  Уникальных рисков: {dataset2['risk_name'].nunique()}")
    
    print(f"\nФОРМАТ КОЛОНКИ 'risk_name' В ДАТАСЕТЕ 2:")
    print(f"  Тип данных: {dataset2['risk_name'].dtype}")
    print(f"  Примеры значений: {dataset2['risk_name'].head(5).tolist()}")
    print(f"  Уникальных значений: {dataset2['risk_name'].nunique()}")
    
    # Анализ целевых переменных для датасета 1
    print("\n" + "=" * 60)
    print("РАСПРЕДЕЛЕНИЕ КЛАССОВ В ЦЕЛЕВЫХ ПЕРЕМЕННЫХ (ДАТАСЕТ 1)")
    print("=" * 60)
    
    target_cols = ['target_source_1', 'target_source_2', 'target_source_3', 'target_source_4']
    for target in target_cols:
        valid_data = dataset1[target].dropna()
        if len(valid_data) > 0:
            counts = valid_data.value_counts().sort_index()
            percentages = valid_data.value_counts(normalize=True).sort_index() * 100
            
            print(f"\n{target}:")
            print(f"  0 (нет событий): {counts.get(0, 0)} ({percentages.get(0, 0):.1f}%)")
            print(f"  1 (есть события): {counts.get(1, 0)} ({percentages.get(1, 0):.1f}%)")
            print(f"  Всего записей: {len(valid_data)}")
            print(f"  Пропусков (последний месяц): {dataset1[target].isna().sum()}")
    
    # Статистика для датасета 2
    print("\n" + "=" * 60)
    print("СТАТИСТИКА ДАТАСЕТА 2")
    print("=" * 60)
    
    if 'target_next_month' in dataset2.columns:
        target_counts = dataset2['target_next_month'].value_counts()
        print(f"Распределение переменной 'наступление риск-события':")
        print(target_counts)
        
        class_dist = dataset2['target_next_month'].value_counts(normalize=True)
        if len(class_dist) == 2:
            print(f"  Дисбаланс классов: {class_dist[0]:.2%} (0 - нет события) vs {class_dist[1]:.2%} (1 - есть событие)")
    
    return dataset1, dataset2

# ИСПОЛЬЗОВАНИЕ ФУНКЦИИ
source_dataset, count_dataset = create_combined_datasets(df, top_bp=20, re_count=100)

ПРЕДВАРИТЕЛЬНАЯ ФИЛЬТРАЦИЯ 
Исходный диапазон дат: 2014-05-20 12:32:04 - 2025-12-17 15:49:08
Данные после фильтрации по периоду 2020-2026: 23646 записей
Диапазон дат после фильтрации: 2020-01-01 00:00:00 - 2025-12-17 15:49:08

Фильтрация бизнес-процессов по условиям...
Бизнес-процессов с более чем 100 событий в общем: 23
Бизнес-процессов с более чем 50 событий в последнем году: 14
Бизнес-процессов, удовлетворяющих обоим условиям: 14

Отобранные бизнес-процессы (14):
  1. 2: всего 659 событий, в последнем году 167
  2. 5: всего 642 событий, в последнем году 262
  3. 6: всего 822 событий, в последнем году 168
  4. 18: всего 402 событий, в последнем году 72
  5. 21: всего 1657 событий, в последнем году 304
  6. 24: всего 135 событий, в последнем году 103
  7. 26: всего 1412 событий, в последнем году 186
  8. 30: всего 3391 событий, в последнем году 468
  9. 32: всего 381 событий, в последнем году 78
  10. 38: всего 1977 событий, в последнем году 321
  11. 39: всего 7479 событий, в последн

In [5]:
# Пример данных для предсказания срабатывания источника риск-события
source_dataset.tail()

,bp_number,risk_name,year_month,risk_rating_encoded,risk_fin_loss_encoded,risk_business_rep_encoded,present_risk_zone_encoded,residual_risk_zone_encoded,consequence_type_encoded,consequence_influence_level_encoded,event_count,total_loss,total_reimburse,source_1_count,source_2_count,source_3_count,source_4_count,source_1_ratio,source_2_ratio,source_3_ratio,source_4_ratio,event_count_lag_1,event_count_lag_2,event_count_lag_3,event_count_rolling_mean_3,event_count_rolling_std_3,event_count_rolling_mean_6,event_count_rolling_std_6,months_since_last_event,month_of_year,source_1_count_lag_1,source_2_count_lag_1,source_3_count_lag_1,source_4_count_lag_1,target_source_1,target_source_2,target_source_3,target_source_4
2906,43,РC-0235,2025-07,2.0,2.0,2.0,1.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,4.666667,8.082904,4.500000,5.282045,3,7,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2907,43,РC-0235,2025-08,2.0,2.0,2.0,1.0,1.0,0.0,1.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,3.500000,5.504544,4,8,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2908,43,РC-0235,2025-09,2.0,2.0,2.0,1.0,1.0,0.0,1.0,6.0,0.0,0.0,6.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0,1.000000,1.732051,3.166667,5.455884,1,9,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2909,43,РC-0235,2025-10,2.0,2.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,3.0,0.0,3.000000,3.000000,3.833333,5.528713,1,10,6.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2910,43,РC-0235,2025-11,2.0,2.0,2.0,1.0,1.0,0.0,1.0,8.0,0.0,0.0,8.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,3.0,3.000000,3.000000,1.500000,2.509980,2,11,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [6]:
# Пример данных для предсказания количества риск-событий
count_dataset.tail()

,bp_number,risk_name,year_month,event_count,risk_rating_encoded,risk_fin_loss_encoded,present_risk_zone_encoded,residual_risk_zone_encoded,consequence_influence_level_encoded,risk_business_rep_encoded,risk_event_source_group_encoded,risk_event_reason_group_encoded,consequence_type_encoded,consequence_kind_encoded,consequence_name_encoded,loss_sum_encoded,reimburse_sum_encoded,event_count_lag_1,event_count_lag_2,event_count_lag_3,event_count_lag_4,event_count_lag_5,event_count_lag_6,event_count_lag_9,event_count_lag_12,event_count_lag_18,event_count_lag_24,event_count_rolling_mean_3,event_count_rolling_std_3,event_count_rolling_mean_6,event_count_rolling_std_6,event_count_rolling_mean_9,event_count_rolling_std_9,event_count_rolling_mean_12,event_count_rolling_std_12,event_count_rolling_mean_18,event_count_rolling_std_18,event_count_rolling_mean_24,event_count_rolling_std_24,target_next_month,event_count_next_month,months_since_last_event,total_events,rolling_mean_3,rolling_std_3,rolling_mean_6,rolling_std_6,rolling_mean_9,rolling_std_9,rolling_mean_12,rolling_std_12,rolling_mean_18,rolling_std_18,rolling_mean_24,rolling_std_24,months_since_first,total_events_so_far,month,quarter,month_sin,month_cos,rating_x_lag1,rating_x_total
2906,43,РC-0235,2025-07,0.0,2,2,1,1,1,2,1,4,1,0,40,0.0,0.0,0.0,0.0,14.0,2.0,5.0,6.0,6.0,0.0,0.0,1.0,0.000000,0.000000,3.500000,5.504544,3.333333,4.609772,3.500000,4.033947,3.666667,4.419209,3.250000,4.386243,1,3,3,134.0,0.000000,0.000000,3.500000,5.504544,3.333333,4.609772,3.500000,4.033947,3.666667,4.419209,3.250000,4.386243,66,117.0,7,3,-0.500000,-8.660254e-01,0.0,234.0
2907,43,РC-0235,2025-08,3.0,2,2,1,1,1,2,1,4,1,0,40,0.0,0.0,0.0,0.0,0.0,14.0,2.0,5.0,0.0,2.0,10.0,0.0,1.000000,1.732051,3.166667,5.455884,3.666667,4.444097,3.583333,4.010403,3.277778,4.127463,3.375000,4.332009,1,6,0,134.0,1.000000,1.732051,3.166667,5.455884,3.666667,4.444097,3.583333,4.010403,3.277778,4.127463,3.375000,4.332009,67,120.0,8,3,-0.866025,-5.000000e-01,0.0,240.0
2908,43,РC-0235,2025-09,6.0,2,2,1,1,1,2,1,4,1,0,40,0.0,0.0,3.0,0.0,0.0,0.0,14.0,2.0,3.0,4.0,12.0,1.0,3.000000,3.000000,3.833333,5.528713,4.000000,4.500000,3.750000,4.070403,2.944444,3.588744,3.583333,4.333055,0,0,0,134.0,3.000000,3.000000,3.833333,5.528713,4.000000,4.500000,3.750000,4.070403,2.944444,3.588744,3.583333,4.333055,68,126.0,9,3,-1.000000,-1.836970e-16,6.0,252.0
2909,43,РC-0235,2025-10,0.0,2,2,1,1,1,2,1,4,1,0,40,0.0,0.0,6.0,3.0,0.0,0.0,0.0,14.0,6.0,6.0,0.0,11.0,3.000000,3.000000,1.500000,2.509980,3.333333,4.609772,3.250000,4.136863,2.944444,3.588744,3.125000,4.089355,1,8,1,134.0,3.000000,3.000000,1.500000,2.509980,3.333333,4.609772,3.250000,4.136863,2.944444,3.588744,3.125000,4.089355,69,126.0,10,4,-0.866025,5.000000e-01,12.0,252.0
2910,43,РC-0235,2025-11,8.0,2,2,1,1,1,2,1,4,1,0,40,0.0,0.0,0.0,6.0,3.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,4.666667,4.163332,2.833333,3.488075,3.666667,4.847680,3.916667,4.209477,3.388889,3.696403,3.458333,4.149166,0,0,0,134.0,4.666667,4.163332,2.833333,3.488075,3.666667,4.847680,3.916667,4.209477,3.388889,3.696403,3.458333,4.149166,70,134.0,11,4,-0.500000,8.660254e-01,0.0,268.0


In [7]:
# датасет для визуализации работы модели
source_part = source_dataset[['bp_number', 'risk_name', 'year_month',
                              'source_1_count', 'source_2_count', 'source_3_count', 'source_4_count']]
count_part = count_dataset[['bp_number', 'risk_name', 'year_month', 'event_count']]

# Выполняем объединение по ключевым полям
base_df = pd.merge(source_part, count_part, on=['bp_number', 'risk_name', 'year_month'], how='inner')

base_df.head()

,bp_number,risk_name,year_month,source_1_count,source_2_count,source_3_count,source_4_count,event_count
0,2,РC-0005,2020-01,0.0,0.0,0.0,0.0,0.0
1,2,РC-0005,2020-02,0.0,0.0,0.0,0.0,0.0
2,2,РC-0005,2020-03,0.0,0.0,0.0,0.0,0.0
3,2,РC-0005,2020-04,0.0,0.0,0.0,0.0,0.0
4,2,РC-0005,2020-05,0.0,0.0,0.0,0.0,0.0


In [8]:
base_df.tail(10)

,bp_number,risk_name,year_month,source_1_count,source_2_count,source_3_count,source_4_count,event_count
2901,43,РC-0235,2025-02,5.0,0.0,0.0,0.0,5.0
2902,43,РC-0235,2025-03,2.0,0.0,0.0,0.0,2.0
2903,43,РC-0235,2025-04,8.0,0.0,3.0,3.0,14.0
2904,43,РC-0235,2025-05,0.0,0.0,0.0,0.0,0.0
2905,43,РC-0235,2025-06,0.0,0.0,0.0,0.0,0.0
2906,43,РC-0235,2025-07,0.0,0.0,0.0,0.0,0.0
2907,43,РC-0235,2025-08,3.0,0.0,0.0,0.0,3.0
2908,43,РC-0235,2025-09,6.0,0.0,0.0,0.0,6.0
2909,43,РC-0235,2025-10,0.0,0.0,0.0,0.0,0.0
2910,43,РC-0235,2025-11,8.0,0.0,0.0,0.0,8.0


# Предсказание моделей

In [15]:
# -------------------------------------------------------------------
# Определение пользовательских классов, использованных при сохранении моделей
# -------------------------------------------------------------------

class WeightedEnsembleRegressor(BaseEstimator, RegressorMixin):
    """Взвешенный ансамбль регрессоров"""
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights if weights is not None else [1/len(models)] * len(models)

    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self

    def predict(self, X):
        predictions = np.array([model.predict(X) for model in self.models])
        weighted_pred = np.average(predictions, axis=0, weights=self.weights)
        return weighted_pred


class ZeroInflatedRegressor(BaseEstimator, RegressorMixin):
    """Zero‑Inflated регрессия"""
    def __init__(self, n_estimators=100, max_depth=10, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.classifier = None
        self.regressor = None

    def fit(self, X, y):
        y_binary = (y > 0).astype(int)
        self.classifier = RandomForestClassifier(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            random_state=self.random_state, n_jobs=-1
        )
        self.classifier.fit(X, y_binary)
        non_zero_idx = y > 0
        if np.sum(non_zero_idx) > 0:
            self.regressor = RandomForestRegressor(
                n_estimators=self.n_estimators, max_depth=self.max_depth,
                random_state=self.random_state, n_jobs=-1
            )
            self.regressor.fit(X[non_zero_idx], y[non_zero_idx])
        return self

    def predict(self, X):
        if hasattr(self.classifier, 'predict_proba'):
            prob_non_zero = self.classifier.predict_proba(X)[:, 1]
        else:
            prob_non_zero = self.classifier.predict(X).astype(float)
        if self.regressor is None:
            return np.zeros(len(X))
        reg_predictions = self.regressor.predict(X)
        predictions = prob_non_zero * reg_predictions
        predictions = np.maximum(predictions, 0)
        predictions = np.round(predictions)
        return predictions


class WeightedProbaEnsemble(BaseEstimator, ClassifierMixin):
    """Взвешенный ансамбль классификаторов по вероятностям """
    def __init__(self, models, weights, threshold=0.5):
        self.models = models
        self.weights = weights
        self.threshold = threshold

    def predict_proba(self, X):
        all_probas = []
        for model in self.models:
            if hasattr(model, 'predict_proba'):
                probas = model.predict_proba(X)[:, 1]
            else:
                probas = model.predict(X).astype(float)
            all_probas.append(probas)
        weighted_probas = np.zeros_like(all_probas[0])
        for i, probas in enumerate(all_probas):
            weighted_probas += probas * self.weights[i]
        prob_array = np.zeros((len(weighted_probas), 2))
        prob_array[:, 1] = weighted_probas
        prob_array[:, 0] = 1 - weighted_probas
        return prob_array

    def predict(self, X):
        probas = self.predict_proba(X)[:, 1]
        return (probas >= self.threshold).astype(int)

    def fit(self, X, y):
        return self


class WeightedEnsemble(BaseEstimator, ClassifierMixin):
    """Взвешенный ансамбль классификаторов """
    def __init__(self, models, weights):
        self.models = models
        self.weights = weights

    def fit(self, X, y):
        for name, model in self.models:
            model.fit(X, y)
        return self

    def predict_proba(self, X):
        probas = np.zeros((len(X), 2))
        probas[:, 1] = np.zeros(len(X))
        for (name, model), weight in zip(self.models, self.weights):
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X)[:, 1]
            else:
                proba = model.predict(X).astype(float)
            probas[:, 1] += weight * proba
        probas[:, 0] = 1 - probas[:, 1]
        return probas

    def predict(self, X, threshold=0.5):
        probas = self.predict_proba(X)[:, 1]
        return (probas >= threshold).astype(int)


# -------------------------------------------------------------------
# Вспомогательные функции предсказания (адаптированы из исходных кодов)
# -------------------------------------------------------------------

def _predict_count(model_data, count_df, prediction_month):
    """Предсказание количества событий """
    classifier = model_data['classifier']
    regressor = model_data['regressor']
    threshold = model_data['final_threshold']
    cls_features = model_data['feature_columns_classifier']
    reg_features = model_data['feature_columns_regressor']

    # Проверка наличия признаков
    missing = set(cls_features) - set(count_df.columns)
    if missing:
        raise ValueError(f"В count_dataset отсутствуют столбцы: {missing}")

    X = count_df[cls_features].copy()
    for col in X.columns:
        if X[col].dtype in ['object', 'period[M]', 'datetime64[ns]']:
            try:
                X[col] = pd.to_numeric(X[col], errors='coerce')
            except:
                X = X.drop(columns=[col])
    X = X.astype(float).fillna(0)

    # Вероятность и бинарное предсказание
    if hasattr(classifier, 'predict_proba'):
        cls_proba = classifier.predict_proba(X)[:, 1]
    else:
        cls_proba = classifier.predict(X).astype(float)
    cls_pred = (cls_proba >= threshold).astype(int)

    # Расширенные признаки
    X_enhanced = X.copy()
    X_enhanced['classifier_proba'] = cls_proba
    X_enhanced['classifier_pred'] = cls_pred
    X_enhanced['proba_squared'] = cls_proba ** 2
    X_enhanced['proba_log'] = np.log1p(cls_proba)

    if 'event_count' in X_enhanced.columns:
        X_enhanced['proba_x_event_count'] = cls_proba * X_enhanced['event_count']
    if 'total_events' in X_enhanced.columns:
        X_enhanced['proba_x_total_events'] = cls_proba * X_enhanced['total_events']

    # Добавление недостающих столбцов
    for col in reg_features:
        if col not in X_enhanced.columns:
            X_enhanced[col] = 0
    X_enhanced = X_enhanced[reg_features]

    # Предсказание регрессии
    y_reg = regressor.predict(X_enhanced)
    y_reg = np.maximum(y_reg, 0)
    y_reg = np.round(y_reg)

    # Обнуление там, где классификатор предсказал 0
    y_reg[cls_pred == 0] = 0
    # Корректировка при высокой вероятности
    high_prob_idx = (y_reg == 0) & (cls_proba > 0.7)
    y_reg[high_prob_idx] = 1

    result = count_df[['bp_number', 'risk_name']].copy()
    result['base_month_count'] = count_df['year_month']
    result['prediction_month'] = prediction_month
    result['predicted_event_count'] = y_reg
    result['event_probability_count'] = cls_proba
    result['classifier_pred'] = cls_pred
    return result


def _predict_sources(model_data, source_df, prediction_month):
    """Предсказание для источников """
    event_model = model_data['event_model']
    source_models = model_data['source_models']
    feature_cols = model_data['feature_columns']
    filter_threshold = model_data.get('filter_threshold', 0.5)
    corr_params = model_data.get('correction_params', {})
    boost_factor = corr_params.get('boost_factor', 2.0)
    min_boost = corr_params.get('min_boost', 1.2)
    max_boost = corr_params.get('max_boost', 4.0)
    penalty_factor = corr_params.get('penalty_factor', 0.5)
    min_penalty = corr_params.get('min_penalty', 0.2)

    missing = set(feature_cols) - set(source_df.columns)
    if missing:
        raise ValueError(f"В source_dataset отсутствуют столбцы: {missing}")

    X = source_df[feature_cols].copy()
    for col in X.columns:
        if X[col].dtype in ['object', 'period[M]', 'datetime64[ns]']:
            try:
                X[col] = pd.to_numeric(X[col], errors='coerce')
            except:
                X = X.drop(columns=[col])
    X = X.astype(float).fillna(0)

    # Вероятность любого события
    if hasattr(event_model, 'predict_proba'):
        event_proba = event_model.predict_proba(X)[:, 1]
    else:
        event_proba = event_model.predict(X).astype(float)

    high_risk = (event_proba >= filter_threshold)

    # Добавляем вероятность как признак
    X_with_event = X.copy()
    X_with_event['event_probability'] = event_proba

    result = source_df[['bp_number', 'risk_name']].copy()
    result['base_month_source'] = source_df['year_month']
    result['prediction_month'] = prediction_month
    result['event_probability_source'] = event_proba
    result['is_high_risk'] = high_risk.astype(int)

    for src in [1, 2, 3, 4]:
        if src in source_models:
            info = source_models[src]
            model = info['best_model']
            thr = info.get('best_threshold', 0.5)

            if hasattr(model, 'predict_proba'):
                raw_proba = model.predict_proba(X_with_event)[:, 1]
            else:
                raw_proba = model.predict(X_with_event).astype(float)

            adjusted = raw_proba.copy()
            corr = np.ones(len(raw_proba))

            for i in range(len(raw_proba)):
                if high_risk[i]:
                    if event_proba[i] >= filter_threshold:
                        excess = (event_proba[i] - filter_threshold) / (1 - filter_threshold + 1e-8)
                        boost = min(max_boost, min_boost + boost_factor * excess)
                        adjusted[i] = min(0.95, raw_proba[i] * boost)
                        corr[i] = boost
                    else:
                        adjusted[i] = min(0.95, raw_proba[i] * min_boost)
                        corr[i] = min_boost
                else:
                    if event_proba[i] < filter_threshold:
                        deficit = (filter_threshold - event_proba[i]) / (filter_threshold + 1e-8)
                        penalty = max(min_penalty, 1 - penalty_factor * deficit)
                        adjusted[i] = raw_proba[i] * penalty
                        corr[i] = penalty
                    else:
                        adjusted[i] = raw_proba[i] * 0.8
                        corr[i] = 0.8

            result[f'prob_source_{src}'] = adjusted
            result[f'prob_source_{src}_pct'] = (adjusted * 100).round(2)
            result[f'pred_source_{src}'] = (adjusted >= thr).astype(int)
            result[f'correction_factor_{src}'] = corr
            result[f'model_source_{src}'] = info.get('best_model_name', 'unknown')
        else:
            result[f'prob_source_{src}'] = 0.0
            result[f'prob_source_{src}_pct'] = 0.0
            result[f'pred_source_{src}'] = 0
            result[f'correction_factor_{src}'] = 1.0
            result[f'model_source_{src}'] = 'none'

    prob_cols = [f'prob_source_{i}' for i in [1, 2, 3, 4]]
    result['pred_prob_sum'] = result[prob_cols].sum(axis=1)
    result['pred_prob_max'] = result[prob_cols].max(axis=1)
    max_col = result[prob_cols].idxmax(axis=1)
    result['pred_main_source'] = max_col.str.replace('prob_source_', '').astype(int)

    return result


# -------------------------------------------------------------------
# Функция для вывода метрик модели количества событий
# -------------------------------------------------------------------
def print_count_metrics(model_data):
    """Вывод метрик модели количества событий из загруженного словаря"""
    print("\n" + "="*60)
    print("МЕТРИКИ МОДЕЛИ КОЛИЧЕСТВА СОБЫТИЙ (predict_count)")
    print("="*60)
    
    # Метрики классификатора (F1, F2 и др.)
    if 'classifier_metrics' in model_data:
        cm = model_data['classifier_metrics']
        print(f"Классификатор (наличие события):")
        print(f"  F1-score:  {cm.get('f1', 'N/A'):.4f}" if cm.get('f1') is not None else "  F1-score:  N/A")
        print(f"  F2-score:  {cm.get('f2', 'N/A'):.4f}" if cm.get('f2') is not None else "  F2-score:  N/A")
        if 'precision' in cm:
            print(f"  Precision: {cm['precision']:.4f}")
        if 'recall' in cm:
            print(f"  Recall:    {cm['recall']:.4f}")
        if 'accuracy' in cm:
            print(f"  Accuracy:  {cm['accuracy']:.4f}")
    
    # Финальные метрики регрессии
    if 'final_metrics' in model_data:
        fm = model_data['final_metrics']
        print(f"\nРегрессия (количество событий):")
        print(f"  SMAPE: {fm.get('smape', 'N/A'):.2f}%" if fm.get('smape') is not None else "  SMAPE: N/A")
        print(f"  MAE:   {fm.get('mae', 'N/A'):.4f}" if fm.get('mae') is not None else "  MAE:   N/A")
        print(f"  RMSE:  {fm.get('rmse', 'N/A'):.4f}" if fm.get('rmse') is not None else "  RMSE:  N/A")
        print(f"  R²:    {fm.get('r2', 'N/A'):.4f}" if fm.get('r2') is not None else "  R²:    N/A")
    
    # Тип лучшей модели классификации
    if 'classifier_name' in model_data:
        print(f"\nЛучшая модель классификации: {model_data['classifier_name']}")
    if 'classifier_type' in model_data:
        print(f"Тип: {model_data['classifier_type']}")
    if 'final_threshold' in model_data:
        print(f"Финальный порог: {model_data['final_threshold']:.4f}")
    print("="*60)


# -------------------------------------------------------------------
# Функция для вывода метрик моделей источников
# -------------------------------------------------------------------
def print_source_metrics(model_data):
    """Вывод метрик моделей для источников из загруженного словаря"""
    print("\n" + "="*60)
    print("МЕТРИКИ МОДЕЛЕЙ ИСТОЧНИКОВ (predict_source)")
    print("="*60)
    
    # Метрики модели события (первый этап)
    if 'metrics' in model_data:
        metrics = model_data['metrics']
        print(f"Модель наличия любого события:")
        print(f"  F2: {metrics.get('event_model_f2', 'N/A'):.4f}")
        print(f"  F1: {metrics.get('event_model_f1', 'N/A'):.4f}")
    
    # Метрики для каждого источника
    if 'source_models' in model_data:
        source_models = model_data['source_models']
        print(f"\nМодели для источников:")
        for src in [1, 2, 3, 4]:
            if src in source_models:
                info = source_models[src]
                print(f"\nИсточник {src}:")
                print(f"  Модель: {info.get('best_model_name', 'unknown')}")
                print(f"  F2: {info.get('best_val_f2', 'N/A'):.4f}")
                print(f"  F1: {info.get('best_val_f1', 'N/A'):.4f}")
                if 'best_metrics' in info:
                    bm = info['best_metrics']
                    print(f"  Precision: {bm.get('precision', 'N/A'):.4f}")
                    print(f"  Recall:    {bm.get('recall', 'N/A'):.4f}")
                print(f"  Порог: {info.get('best_threshold', 'N/A'):.4f}")
                if info.get('is_ensemble', False):
                    print(f"  (ансамбль, тип: {info.get('ensemble_type', 'unknown')})")
    
    # Фильтр для первого этапа
    if 'filter_threshold' in model_data:
        print(f"\nПорог фильтрации (is_high_risk): {model_data['filter_threshold']:.4f}")
    print("="*60)


# -------------------------------------------------------------------
# Основная функция для предсказания на заданный месяц
# -------------------------------------------------------------------
def predict_for_month(target_month: str,
                      count_df: pd.DataFrame,
                      source_df: pd.DataFrame,
                      count_model_path: str = 'predict_count.pkl',
                      source_model_path: str = 'predict_source.pkl',
                      true_count_df: pd.DataFrame = None,
                      true_source_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Предсказывает количество событий и вероятности источников на указанный месяц.

    Алгоритм:
    1. По целевому месяцу определяется предыдущий месяц (данные за который нужны для признаков).
    2. Из переданных датафреймов выбираются записи за этот предыдущий месяц.
       Если такого месяца нет, берётся последний доступный месяц, предшествующий целевому,
       и выдаётся предупреждение.
    3. Загружаются обученные модели и выводятся их сохранённые метрики.
    4. Выполняются предсказания количества событий и для источников.
    5. Если переданы истинные значения (true_count_df, true_source_df) за целевой месяц,
       вычисляются и выводятся метрики сравнения предсказаний с фактом.
    6. Результаты объединяются по `bp_number` и `risk_name` и возвращаются.

    Parameters
    ----------
    target_month : str
        Месяц, на который делается предсказание, в формате 'YYYY-MM' (например, '2025-08').
    count_df : pd.DataFrame
        Датасет для модели количества событий. Должен содержать столбец 'year_month'
        и все признаки, необходимые модели.
    source_df : pd.DataFrame
        Датасет для модели источников. Должен содержать столбец 'year_month'
        и все признаки, необходимые модели.
    count_model_path : str, default='predict_count.pkl'
        Путь к файлу сохранённой модели количества событий.
    source_model_path : str, default='predict_source.pkl'
        Путь к файлу сохранённой модели источников.
    true_count_df : pd.DataFrame, optional
        Датафрейм с фактическими значениями за целевой месяц для модели количества.
        Должен содержать столбцы 'bp_number', 'risk_name', 'event_count_next_month' (или аналогичный).
    true_source_df : pd.DataFrame, optional
        Датафрейм с фактическими значениями за целевой месяц для источников.
        Должен содержать столбцы 'bp_number', 'risk_name', 'target_source_1'..'target_source_4'.

    Returns
    -------
    pd.DataFrame
        Датафрейм с предсказаниями на `target_month`. Содержит колонки:
        - bp_number, risk_name
        - base_month_count, base_month_source (фактические месяцы, по которым делалось предсказание)
        - prediction_month (целевой месяц)
        - predicted_event_count, event_probability_count, classifier_pred (из модели количества)
        - event_probability_source, is_high_risk, prob_source_1..4, prob_source_1_pct..4_pct,
          pred_source_1..4, correction_factor_1..4, model_source_1..4,
          pred_prob_sum, pred_prob_max, pred_main_source (из модели источников)
    """
    # Определяем предыдущий месяц относительно target_month
    try:
        target_dt = pd.to_datetime(target_month)
    except:
        raise ValueError("target_month должен быть в формате 'YYYY-MM'")

    prev_month_dt = target_dt - pd.DateOffset(months=1)
    prev_month_str = prev_month_dt.strftime('%Y-%m')

    # Функция для безопасного получения данных за нужный месяц
    def get_data_for_month(df, month_str, dataset_name):
        mask = df['year_month'] == month_str
        if mask.any():
            return df[mask].copy()
        else:
            # Если точного месяца нет, берём последний доступный месяц < target_month
            available_months = pd.to_datetime(df['year_month'].unique())
            available_before = available_months[available_months < target_dt]
            if len(available_before) == 0:
                raise ValueError(f"В {dataset_name} нет данных до {target_month}")
            last_avail = available_before.max().strftime('%Y-%m')
            warnings.warn(f"В {dataset_name} нет данных за {month_str}. "
                          f"Используются данные за {last_avail}.")
            return df[df['year_month'] == last_avail].copy()

    count_input = get_data_for_month(count_df, prev_month_str, 'count_dataset')
    source_input = get_data_for_month(source_df, prev_month_str, 'source_dataset')

    print(f"\nПредсказание на {target_month} выполняется на основе данных за "
          f"{count_input['year_month'].iloc[0]} (count) и "
          f"{source_input['year_month'].iloc[0]} (source)")

    # Загрузка моделей
    print("\nЗагрузка моделей...")
    count_model_data = joblib.load(count_model_path)
    source_model_data = joblib.load(source_model_path)

    # Вывод метрик загруженных моделей
    print_count_metrics(count_model_data)
    print_source_metrics(source_model_data)

    # Получение предсказаний
    count_pred = _predict_count(count_model_data, count_input, target_month)
    source_pred = _predict_sources(source_model_data, source_input, target_month)

    # Объединение
    merged = pd.merge(
        count_pred,
        source_pred,
        on=['bp_number', 'risk_name'],
        how='outer',
        suffixes=('_count', '_source')
    )

    # Унифицируем колонку prediction_month
    if 'prediction_month_count' in merged.columns:
        merged['prediction_month'] = merged['prediction_month_count']
        merged.drop(columns=['prediction_month_count', 'prediction_month_source'], errors='ignore', inplace=True)
    elif 'prediction_month_source' in merged.columns:
        merged['prediction_month'] = merged['prediction_month_source']
        merged.drop(columns=['prediction_month_source'], errors='ignore', inplace=True)

    # Если переданы истинные значения, оцениваем качество предсказаний
    if true_count_df is not None or true_source_df is not None:
        print("\n" + "="*60)
        print("ОЦЕНКА КАЧЕСТВА ПРЕДСКАЗАНИЙ НА ЦЕЛЕВОЙ МЕСЯЦ")
        print("="*60)

        # Метрики для количества событий
        if true_count_df is not None:
            true_count = true_count_df[true_count_df['year_month'] == target_month]
            if not true_count.empty:
                # Объединяем с предсказаниями
                eval_count = pd.merge(
                    merged[['bp_number', 'risk_name', 'predicted_event_count']],
                    true_count[['bp_number', 'risk_name', 'event_count_next_month']],
                    on=['bp_number', 'risk_name'],
                    how='inner'
                )
                if not eval_count.empty:
                    y_true = eval_count['event_count_next_month'].values
                    y_pred = eval_count['predicted_event_count'].values

                    # SMAPE
                    denominator = np.abs(y_true) + np.abs(y_pred) + 1e-10
                    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / denominator)

                    print(f"\nМетрики для количества событий (на {len(eval_count)} записях):")
                    print(f"  SMAPE: {smape:.2f}%")
                    print(f"  MAE:   {mean_absolute_error(y_true, y_pred):.4f}")
                    print(f"  RMSE:  {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
                    print(f"  R²:    {r2_score(y_true, y_pred):.4f}")
                else:
                    print("\nНет совпадений для оценки модели количества событий.")
            else:
                print(f"\nВ true_count_df нет данных за {target_month}.")

        # Метрики для источников
        if true_source_df is not None:
            true_src = true_source_df[true_source_df['year_month'] == target_month]
            if not true_src.empty:
                # Собираем предсказания бинарных меток для источников
                eval_src = pd.merge(
                    merged[['bp_number', 'risk_name'] + [f'pred_source_{i}' for i in [1,2,3,4]]],
                    true_src[['bp_number', 'risk_name'] + [f'target_source_{i}' for i in [1,2,3,4]]],
                    on=['bp_number', 'risk_name'],
                    how='inner'
                )
                if not eval_src.empty:
                    print(f"\nМетрики для источников (на {len(eval_src)} записях):")
                    for src in [1,2,3,4]:
                        y_true = eval_src[f'target_source_{src}'].values
                        y_pred = eval_src[f'pred_source_{src}'].values
                        if len(np.unique(y_true)) > 1:
                            f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
                            f1 = fbeta_score(y_true, y_pred, beta=1, zero_division=0)
                            prec = precision_score(y_true, y_pred, zero_division=0)
                            rec = recall_score(y_true, y_pred, zero_division=0)
                            print(f"\nИсточник {src}:")
                            print(f"  F2: {f2:.4f}")
                            print(f"  F1: {f1:.4f}")
                            print(f"  Precision: {prec:.4f}")
                            print(f"  Recall:    {rec:.4f}")
                        else:
                            print(f"\nИсточник {src}: только один класс в истинных значениях.")
                else:
                    print("\nНет совпадений для оценки моделей источников.")
            else:
                print(f"\nВ true_source_df нет данных за {target_month}.")

    # Упорядочивание колонок
    base_cols = ['bp_number', 'risk_name', 'base_month_count', 'base_month_source', 'prediction_month']
    count_cols = ['predicted_event_count', 'event_probability_count', 'classifier_pred']
    source_cols = ['event_probability_source', 'is_high_risk',
                   'prob_source_1_pct', 'prob_source_2_pct', 'prob_source_3_pct', 'prob_source_4_pct',
                   'pred_source_1', 'pred_source_2', 'pred_source_3', 'pred_source_4']
    all_cols = base_cols + count_cols + source_cols
    existing = [c for c in all_cols if c in merged.columns]
    merged = merged[existing]

    return merged


# -------------------------------------------------------------------
# Пример использования
# -------------------------------------------------------------------
if __name__ == '__main__':
    target_month = '2025-09'

    # Вызов функции
    result = predict_for_month(
        target_month=target_month,
        count_df=count_dataset,
        source_df=source_dataset,
        count_model_path='predict_count.pkl',
        source_model_path='predict_source.pkl'
    )

    # Сохранение результата
    output_file = 'predictions.csv'
    result.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nРезультат сохранён в {output_file}")

    # Краткая статистика по предсказаниям
    print("\n" + "="*60)
    print("СТАТИСТИКА ПОЛУЧЕННЫХ ПРЕДСКАЗАНИЙ")
    print("="*60)
    print(f"Всего записей: {len(result)}")
    if 'predicted_event_count' in result.columns:
        print(f"Среднее predicted_event_count: {result['predicted_event_count'].mean():.2f}")
        print(f"Медиана: {result['predicted_event_count'].median():.2f}")
        print(f"Максимум: {result['predicted_event_count'].max()}")
        print(f"Доля нулевых предсказаний: {(result['predicted_event_count'] == 0).mean()*100:.1f}%")
    if 'pred_main_source' in result.columns:
        print(f"\nРаспределение основного источника:")
        print(result['pred_main_source'].value_counts().sort_index())
    print("="*60)


Предсказание на 2025-09 выполняется на основе данных за 2025-08 (count) и 2025-08 (source)

Загрузка моделей...

МЕТРИКИ МОДЕЛИ КОЛИЧЕСТВА СОБЫТИЙ (predict_count)
Классификатор (наличие события):
  F1-score:  0.8686
  F2-score:  0.9087

Регрессия (количество событий):
  SMAPE: 23.51%
  MAE:   1.5567
  RMSE:  4.2483
  R²:    0.7924

Лучшая модель классификации: EasyEnsemble
Тип: single
Финальный порог: 0.7000

МЕТРИКИ МОДЕЛЕЙ ИСТОЧНИКОВ (predict_source)
Модель наличия любого события:
  F2: 0.8077
  F1: 0.7167

Модели для источников:

Источник 1:
  Модель: Ensemble_voting
  F2: 0.8863
  F1: 0.8480
  Precision: 0.7910
  Recall:    0.9138
  Порог: 0.4240
  (ансамбль, тип: voting)

Источник 2:
  Модель: Ensemble_voting
  F2: 0.8834
  F1: 0.8696
  Precision: 0.8475
  Recall:    0.8929
  Порог: 0.4102
  (ансамбль, тип: voting)

Источник 3:
  Модель: Ensemble_weighted
  F2: 0.8186
  F1: 0.7558
  Precision: 0.6701
  Recall:    0.8667
  Порог: 0.4066
  (ансамбль, тип: weighted)

Источник 4:
  М

In [11]:
result.head(10)

,bp_number,risk_name,base_month_count,base_month_source,prediction_month,predicted_event_count,event_probability_count,classifier_pred,event_probability_source,is_high_risk,prob_source_1_pct,prob_source_2_pct,prob_source_3_pct,prob_source_4_pct,pred_source_1,pred_source_2,pred_source_3,pred_source_4
0,2,РC-0005,2025-08,2025-08,2025-09,0.0,0.367439,0,0.463100,0,21.09,6.57,28.89,15.03,0,0,0,0
1,2,РC-0006,2025-08,2025-08,2025-09,0.0,0.565686,0,0.422243,0,5.52,3.14,14.71,1.64,0,0,0,0
2,5,РC-0012,2025-08,2025-08,2025-09,0.0,0.455348,0,0.457823,0,45.68,33.44,46.94,40.77,1,0,1,1
3,5,РC-0014,2025-08,2025-08,2025-09,0.0,0.373368,0,0.351781,0,1.73,2.51,9.75,2.22,0,0,0,0
4,6,РC-0017,2025-08,2025-08,2025-09,0.0,0.454101,0,0.380529,0,2.08,2.24,11.48,5.87,0,0,0,0
5,6,РC-0018,2025-08,2025-08,2025-09,0.0,0.417858,0,0.532813,1,31.88,30.23,39.92,39.08,0,0,0,1
6,6,РC-0020,2025-08,2025-08,2025-09,0.0,0.518359,0,0.558429,1,58.26,51.59,73.94,81.60,1,1,1,1
7,18,РC-0055,2025-08,2025-08,2025-09,0.0,0.267432,0,0.395140,0,21.97,13.87,20.39,40.05,0,0,0,1
8,21,РC-0066,2025-08,2025-08,2025-09,0.0,0.126342,0,0.199607,0,23.50,10.57,20.13,21.86,0,0,0,0
9,21,РC-0067,2025-08,2025-08,2025-09,0.0,0.388233,0,0.517892,1,32.91,31.12,51.29,69.80,0,0,1,1


In [16]:
result = pd.merge(result, base_df['year_month'], left_on = ['risk_name', 'prediction_month'], right_on = ['risk_name', 'year_month'] )
result.head()

KeyError: 'risk_name'

In [ ]:
base_df

In [ ]:
base_df.to_csv('vizualization.csv', index=False, encoding='utf-8-sig')
result.to_csv(output_file, index=False, encoding='utf-8-sig')

# Визуализация

In [ ]:

def improved_smape(y_true, y_pred):
    """Улучшенный SMAPE с защитой от деления на ноль"""
    denominator = np.abs(y_true) + np.abs(y_pred) + 1e-10
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denominator)

class WeightedEnsembleRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights if weights is not None else [1/len(models)] * len(models)
        
    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self
    
    def predict(self, X):
        predictions = [model.predict(X) for model in self.models]
        return np.average(predictions, axis=0, weights=self.weights)

class TwoStageRiskPredictor:
    def __init__(self):
        self.classifier = None
        self.regressor = None
    
    def create_features(self, data, model_data=None):
        data = data.copy()
        if model_data and 'feature_columns_classifier' in model_data:
            feature_cols = model_data['feature_columns_classifier']
            available_cols = [col for col in feature_cols if col in data.columns]
            data = data[available_cols]
            for col in feature_cols:
                if col not in data.columns:
                    data[col] = 0
        else:
            exclude_cols = ['event_count_next_month', 'has_event_next_month', 
                           'log_event_count', 'target_next_month', 'year_month']
            data = data.drop(columns=[col for col in exclude_cols if col in data.columns])
        
        for col in data.columns:
            if data[col].dtype in ['object', 'period[M]', 'datetime64[ns]']:
                try:
                    data[col] = pd.to_numeric(data[col], errors='coerce')
                except:
                    data = data.drop(columns=[col])
        
        return data.astype(float).fillna(0)
    
    def create_enhanced_features(self, X, classifier_proba, classifier_pred):
        X_enhanced = X.copy()
        X_enhanced['classifier_proba'] = classifier_proba
        X_enhanced['classifier_pred'] = classifier_pred
        X_enhanced['proba_squared'] = classifier_proba ** 2
        X_enhanced['proba_log'] = np.log1p(classifier_proba)
        
        if 'event_count' in X_enhanced.columns:
            X_enhanced['proba_x_event_count'] = classifier_proba * X_enhanced['event_count']
        if 'total_events' in X_enhanced.columns:
            X_enhanced['proba_x_total_events'] = classifier_proba * X_enhanced['total_events']
            
        return X_enhanced

def split_data_by_date(data, train_end_date, test_date):
    """Разделение данных по датам"""
    data = data.copy()
    data['year_month_str'] = data['year_month'].astype(str)
    
    train_data = data[data['year_month_str'] <= str(train_end_date)].copy()
    test_data = data[data['year_month_str'] == str(test_date)].copy()
    
    train_data.drop(columns=['year_month_str'], errors='ignore', inplace=True)
    test_data.drop(columns=['year_month_str'], errors='ignore', inplace=True)
    
    print(f"Обучающая выборка: {len(train_data)} записей (до {train_end_date})")
    print(f"Тестовая выборка: {len(test_data)} записей ({test_date})")
    
    return train_data, test_data

def predict_for_month(train_data, test_data, model_data, predictor, target_month):
    """Предсказание на заданный месяц"""
    print(f"\nПредсказание на месяц: {target_month}")
    print("="*60)
    
    unique_combinations = train_data[['bp_number', 'risk_name']].drop_duplicates()
    classifier = model_data['classifier']
    ensemble_regressor = model_data['ensemble_regressor']
    best_threshold = model_data.get('best_threshold', 0.5)
    feature_columns_classifier = model_data['feature_columns_classifier']
    
    feature_columns_regressor = model_data.get('feature_columns_regressor') or \
                               model_data.get('X_test_enhanced_columns') or \
                               (feature_columns_classifier + [
                                   'classifier_proba', 'classifier_pred', 
                                   'proba_squared', 'proba_log',
                                   'proba_x_event_count', 'proba_x_total_events'
                               ])
    
    all_predictions = []
    
    for idx, (bp_num, risk_name) in enumerate(unique_combinations.itertuples(index=False), 1):
        combo_mask = (train_data['bp_number'] == bp_num) & (train_data['risk_name'] == risk_name)
        combo_data = train_data[combo_mask].copy()
        
        if len(combo_data) == 0:
            continue
        
        last_record = combo_data.iloc[-1].copy()
        last_record['year_month'] = target_month
        future_data = pd.DataFrame([last_record])
        
        X_future = predictor.create_features(future_data, model_data)
        
        for col in feature_columns_classifier:
            if col not in X_future.columns:
                X_future[col] = 0
        
        X_future = X_future[feature_columns_classifier].astype(float)
        
        if hasattr(classifier, 'predict_proba'):
            y_cls_proba = classifier.predict_proba(X_future)[:, 1]
        else:
            y_cls_proba = classifier.predict(X_future)
        
        y_cls_pred = (y_cls_proba >= best_threshold).astype(int)
        y_reg_pred = 0
        
        if y_cls_pred[0] == 1:
            X_future_enhanced = predictor.create_enhanced_features(
                X_future, y_cls_proba[0], y_cls_pred[0]
            )
            
            for col in feature_columns_regressor:
                if col not in X_future_enhanced.columns:
                    X_future_enhanced[col] = 0
            
            X_future_enhanced = X_future_enhanced[feature_columns_regressor].astype(float)
            y_reg_pred = ensemble_regressor.predict(X_future_enhanced)[0]
        
        y_reg_pred = max(y_reg_pred, 0)
        y_reg_pred = round(y_reg_pred)
        
        if y_reg_pred == 0 and y_cls_proba[0] > 0.7:
            y_reg_pred = 1
        elif y_reg_pred > 0 and y_cls_proba[0] < 0.3:
            y_reg_pred = 0
        
        actual_event_count = 0
        actual_data = test_data[(test_data['bp_number'] == bp_num) & 
                               (test_data['risk_name'] == risk_name)]
        if not actual_data.empty:
            actual_event_count = actual_data['event_count'].iloc[0]   # изменено: было event_count_next_month
        
        error = y_reg_pred - actual_event_count
        
        all_predictions.append({
            'bp_number': bp_num,
            'risk_name': risk_name,
            'year_month': target_month,
            'predicted_event_count': y_reg_pred,
            'actual_event_count': actual_event_count,
            'prediction_error': error,
            'absolute_error': abs(error),
            'predicted_probability': float(y_cls_proba[0]),
            'prediction_correct': (y_reg_pred > 0) == (actual_event_count > 0),
            'has_predicted_event': y_reg_pred > 0,
            'has_actual_event': actual_event_count > 0
        })
        
        if idx % 100 == 0:
            print(f"Обработано {idx}/{len(unique_combinations)} комбинаций...")
    
    predictions_df = pd.DataFrame(all_predictions)
    
    # Расчет метрик
    if len(predictions_df) > 0:
        smape = improved_smape(predictions_df['actual_event_count'].values, 
                               predictions_df['predicted_event_count'].values)
        mae = predictions_df['absolute_error'].mean()
        accuracy = predictions_df['prediction_correct'].mean()
        
        tp = ((predictions_df['has_predicted_event']) & (predictions_df['has_actual_event'])).sum()
        fp = ((predictions_df['has_predicted_event']) & (~predictions_df['has_actual_event'])).sum()
        fn = ((~predictions_df['has_predicted_event']) & (predictions_df['has_actual_event'])).sum()
        tn = ((~predictions_df['has_predicted_event']) & (~predictions_df['has_actual_event'])).sum()
        
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1 = 2 * precision * recall / (precision + recall + 1e-10) if (precision + recall) > 0 else 0
        
        # Расчет RMSE
        rmse = np.sqrt(np.mean((predictions_df['predicted_event_count'] - predictions_df['actual_event_count'])**2))
        
        print(f"\n{'='*60}")
        print("МЕТРИКИ КАЧЕСТВА МОДЕЛИ:")
        print(f"{'='*60}")
        print(f"SMAPE: {smape:.1f}%")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1-Score: {f1:.3f}")
        print(f"\nМатрица ошибок: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
        print(f"\nПредсказано событий: {predictions_df['predicted_event_count'].sum():.0f}")
        print(f"Фактических событий: {predictions_df['actual_event_count'].sum():.0f}")
        print(f"Общее количество комбинаций: {len(predictions_df)}")
        print(f"{'='*60}")
    
    return predictions_df

def visualize_bp_risks_comparison(predictions_df, bp_number, figsize=(16, 10)):
    """
    График сравнения предсказанных и фактических событий по всем рискам для выбранного бизнес-процесса
    """
    # Преобразуем bp_number к тому же типу, что и в данных
    bp_predictions = predictions_df[predictions_df['bp_number'].astype(str) == str(bp_number)].copy()
    
    if len(bp_predictions) == 0:
        print(f"Нет данных для бизнес-процесса {bp_number}")
        return None
    
    # Сортируем риски по предсказанному количеству событий
    bp_predictions = bp_predictions.sort_values('predicted_event_count', ascending=False)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    risks = bp_predictions['risk_name'].values
    y_pos = np.arange(len(risks))
    predicted = bp_predictions['predicted_event_count'].astype(int).values
    actual = bp_predictions['actual_event_count'].astype(int).values
    width = 0.35
    
    # График: Предсказанные vs Фактические события
    ax.barh(y_pos - width/2, predicted, width, 
            color='red', alpha=0.7, edgecolor='black', 
            label='Предсказано')
    ax.barh(y_pos + width/2, actual, width, 
            color='green', alpha=0.7, edgecolor='black',
            label='Факт')
    
    # Добавляем значения на столбцы
    for i, (p, a) in enumerate(zip(predicted, actual)):
        if p > 0:
            ax.text(p + 0.1, i - width/2, str(p), va='center', fontsize=8)
        if a > 0:
            ax.text(a + 0.1, i + width/2, str(a), va='center', fontsize=8)
    
    ax.set_yticks(y_pos)
    # Преобразуем все названия рисков в строки для отображения
    risk_labels = [str(risk) for risk in risks]
    ax.set_yticklabels(risk_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Количество событий', fontsize=12)
    ax.set_title(f'Бизнес-процесс {bp_number}: Предсказанные vs Фактические события по рискам', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Устанавливаем единую шкалу X с шагом 5
    max_value = max(np.max(predicted), np.max(actual))
    x_max = ((max_value // 5) + 1) * 5  # Округляем до ближайшего кратного 5
    ax.set_xlim(0, x_max)
    ax.set_xticks(np.arange(0, x_max + 1, 5))
    
    plt.tight_layout()
    plt.show()
    
    # Выводим статистику
    print(f"\nСтатистика для бизнес-процесса {bp_number}:")
    print(f"{'='*60}")
    print(f"Количество рисков: {len(bp_predictions)}")
    print(f"Всего предсказано событий: {bp_predictions['predicted_event_count'].sum():.0f}")
    print(f"Всего фактических событий: {bp_predictions['actual_event_count'].sum():.0f}")
    print(f"Средняя абсолютная ошибка (MAE): {bp_predictions['absolute_error'].mean():.2f}")
    print(f"SMAPE: {improved_smape(bp_predictions['actual_event_count'].values, bp_predictions['predicted_event_count'].values):.1f}%")
    
    return bp_predictions

def visualize_risk_time_series(data, predictions_df, bp_number, risk_name, months_to_show=12):
    """
    Временной ряд для конкретного риска с предсказанным и фактическим количеством событий.
    Добавлена информация о вероятностях срабатывания источников под графиком.
    """
    # Преобразуем risk_name в строку для безопасной обработки
    risk_name_str = str(risk_name)
    
    print(f"\nВременной ряд для риска: {risk_name_str} (БП: {bp_number})")
    print("="*60)
    
    # Фильтруем данные для конкретной комбинации
    combo_mask = (data['bp_number'].astype(str) == str(bp_number)) & (data['risk_name'] == risk_name)
    combo_data = data[combo_mask].copy()
    
    if len(combo_data) == 0:
        print(f"Данные не найдены для BP {bp_number}, риск {risk_name_str}")
        return None
    
    # Получаем предсказание
    pred_mask = (predictions_df['bp_number'].astype(str) == str(bp_number)) & (predictions_df['risk_name'] == risk_name)
    if not pred_mask.any():
        print(f"Нет предсказания для BP {bp_number}, риск {risk_name_str}")
        return None
    
    prediction = predictions_df[pred_mask].iloc[0]
    predicted_count = int(prediction['predicted_event_count'])
    actual_count = int(prediction['actual_event_count'])
    predicted_prob = float(prediction['predicted_probability'])
    target_month = str(prediction['year_month'])
    
    # Получаем данные по источникам (вероятности и предсказания)
    source_probs = []
    source_preds = []
    for i in range(1, 5):
        prob_col = f'prob_source_{i}_pct'
        pred_col = f'pred_source_{i}'
        source_probs.append(float(prediction.get(prob_col, 0.0)))
        source_preds.append(int(prediction.get(pred_col, 0)))
    
    # Получаем фактические значения источников за месяц предсказания (из prediction, добавлены при слиянии)
    source_actual = []
    for i in range(1, 5):
        source_col = f'source_{i}_count'
        source_actual.append(int(prediction.get(source_col, 0)))
    
    # Ограничиваем количество показываемых месяцев
    if len(combo_data) > months_to_show:
        combo_data = combo_data.tail(months_to_show)
    
    # Создаем график
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Подготовка данных для графика
    dates = []
    for date_val in combo_data['year_month']:
        if hasattr(date_val, 'strftime'):
            dates.append(date_val.strftime('%Y-%m'))
        else:
            dates.append(str(date_val))
    
    actual_values = combo_data['event_count'].astype(int).values
    dates_with_pred = dates + [target_month]
    x_ticks = np.arange(len(dates_with_pred))
    historical_with_pred = list(actual_values) + [np.nan]
    predicted_vals = [np.nan] * len(dates) + [predicted_count]
    actual_vals = [np.nan] * len(dates) + [actual_count]
    
    # График: Исторические данные и предсказание
    ax.plot(x_ticks[:-1], historical_with_pred[:-1], 
             marker='o', markersize=8, linewidth=2, 
             label='Исторические события', color='#2E86AB', alpha=0.8, zorder=2)
    
    # Цвета для предсказания и факта
    actual_color = 'green' if actual_count > 0 else 'lightgreen'
    
    # Добавляем предсказание и факт
    ax.scatter(x_ticks[-1], predicted_vals[-1], 
                s=200, color='red', edgecolors='black', linewidth=2,
                label=f'Предсказание на {target_month}', zorder=3, marker='s')
    
    ax.scatter(x_ticks[-1], actual_vals[-1], 
                s=200, color=actual_color, edgecolors='black', linewidth=2,
                label=f'Факт на {target_month}', zorder=4, marker='^')
    
    # Соединяем последнюю историческую точку с предсказанием и фактом
    if len(historical_with_pred) > 1:
        last_historical = historical_with_pred[-2]
        ax.plot([x_ticks[-2], x_ticks[-1]], [last_historical, predicted_vals[-1]], 
                 linestyle='--', color='red', alpha=0.5, linewidth=1.5, zorder=1)
        ax.plot([x_ticks[-2], x_ticks[-1]], [last_historical, actual_vals[-1]], 
                 linestyle=':', color=actual_color, alpha=0.5, linewidth=1.5, zorder=1)
    
    ax.set_title(f'Бизнес-процесс {bp_number} · Риск {risk_name_str}', 
                  fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Месяц', fontsize=12)
    ax.set_ylabel('Количество событий', fontsize=12)
    ax.legend(loc='upper left', fontsize=11)
    ax.grid(True, alpha=0.3, zorder=0)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(dates_with_pred, rotation=45, ha='right')
    
    # Добавляем вероятность события над красным квадратом
    if predicted_prob > 0:
        y_min, y_max = ax.get_ylim()
        text_height = (y_max - y_min) * 0.05
        text_y = predicted_vals[-1] + text_height
        if text_y > y_max:
            ax.set_ylim(y_min, text_y + text_height)
            y_max = text_y + text_height
        ax.text(x_ticks[-1], text_y,
                 f'Вероятность: {predicted_prob:.1%}',
                 fontsize=10, ha='center', va='bottom',
                 bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    # --- Подробная легенда об источниках под графиком ---
    legend_lines = []
    legend_lines.append(f"Источники за {target_month} (Да/Нет – сработал ли по предсказанию):")
    for i in range(4):
        status_text = "Да" if source_preds[i] == 1 else "Нет"
        legend_lines.append(f"  И{i+1}: {source_probs[i]:.1f}% ({status_text}), факт: {source_actual[i]}")
    legend_text = "\n".join(legend_lines)

    # Размещаем под графиком по центру
    ax.text(0.5, -0.15, legend_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black'))

    # Увеличиваем нижний отступ, чтобы текст не налезал на подписи
    plt.subplots_adjust(bottom=0.2)
    
    # Выводим статистику в консоль
    total_historical = combo_data['event_count'].sum()
    avg_historical = combo_data['event_count'].mean()
    max_historical = combo_data['event_count'].max()
    
    print(f"Статистика риска '{risk_name_str}' (БП {bp_number}):")
    print(f"  • Всего исторических записей: {len(combo_data)}")
    print(f"  • Период: {dates[0]} - {dates[-1]}")
    print(f"  • Всего событий в истории: {total_historical:.0f}")
    print(f"  • Среднее в месяц: {avg_historical:.2f}")
    print(f"  • Максимум в месяц: {max_historical:.0f}")
    print(f"  • На {target_month}:")
    print(f"    - Предсказано событий: {predicted_count:.0f}")
    print(f"    - Фактических событий: {actual_count:.0f}")
    print(f"    - Вероятность события: {predicted_prob:.1%}")
    print(f"    - Ошибка предсказания: {predicted_count - actual_count:+d}")
    print("  • Источники (факт за месяц):")
    for i in range(4):
        status = "сработал" if source_preds[i] == 1 else "не сработал"
        print(f"    - Источник {i+1}: {source_probs[i]:.1f}% ({status}), факт: {source_actual[i]}")
    
    plt.tight_layout()
    plt.show()
    
    return combo_data

def visualize_bp_time_series(data, predictions_df, bp_number, months_to_show=24):
    """
    Временной ряд всех риск-событий по бизнес-процессу + предсказанное суммарное количество
    """
    print(f"\nСуммарный временной ряд для бизнес-процесса {bp_number}")
    print("="*60)
    
    # Фильтруем данные по бизнес-процессу
    bp_data = data[data['bp_number'].astype(str) == str(bp_number)].copy()
    
    if len(bp_data) == 0:
        print(f"Нет данных для бизнес-процесса {bp_number}")
        return None
    
    # Группируем по месяцам для получения суммарных событий
    bp_data['year_month_str'] = bp_data['year_month'].astype(str)
    monthly_totals = bp_data.groupby('year_month_str')['event_count'].sum().reset_index()
    monthly_totals = monthly_totals.sort_values('year_month_str')
    
    # Ограничиваем количество месяцев
    if len(monthly_totals) > months_to_show:
        monthly_totals = monthly_totals.tail(months_to_show)
    
    # Получаем предсказания для этого бизнес-процесса
    bp_predictions = predictions_df[predictions_df['bp_number'].astype(str) == str(bp_number)]
    
    if len(bp_predictions) > 0:
        target_month = str(bp_predictions['year_month'].iloc[0])
        total_predicted = bp_predictions['predicted_event_count'].sum()
        total_actual = bp_predictions['actual_event_count'].sum()
    else:
        target_month = None
        total_predicted = 0
        total_actual = 0
    
    # Создаем график
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # График: Временной ряд суммарных событий
    dates = monthly_totals['year_month_str'].values
    x_ticks = np.arange(len(dates))
    values = monthly_totals['event_count'].values
    
    ax.plot(x_ticks, values, 
            marker='o', markersize=6, linewidth=2, 
            color='#2E86AB', alpha=0.8, label='Суммарные события')
    
    # Добавляем предсказание на следующий месяц
    if target_month and target_month not in dates:
        x_pred = len(dates)
        ax.scatter(x_pred, total_predicted, 
                  s=200, color='red', edgecolors='black', linewidth=2,
                  label=f'Предсказание на {target_month}', zorder=5, marker='s')
        ax.scatter(x_pred, total_actual, 
                  s=200, color='green', edgecolors='black', linewidth=2,
                  label=f'Факт на {target_month}', zorder=5, marker='^')
        
        if len(values) > 0:
            last_value = values[-1]
            ax.plot([x_ticks[-1], x_pred], [last_value, total_predicted], 
                   linestyle='--', color='red', alpha=0.5, linewidth=1.5)
            ax.plot([x_ticks[-1], x_pred], [last_value, total_actual], 
                   linestyle=':', color='green', alpha=0.5, linewidth=1.5)
        
        all_dates = list(dates) + [target_month]
        all_x_ticks = np.arange(len(all_dates))
        ax.set_xticks(all_x_ticks)
        ax.set_xticklabels(all_dates, rotation=45, ha='right')
    else:
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(dates, rotation=45, ha='right')
    
    ax.set_title(f'Бизнес-процесс {bp_number}: Суммарные риск-события', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Месяц', fontsize=14)
    ax.set_ylabel('Количество событий', fontsize=14)
    ax.legend(loc='upper left', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nСуммарная статистика по бизнес-процессу {bp_number}:")
    print(f"{'='*60}")
    print(f"Всего месяцев в истории: {len(monthly_totals)}")
    print(f"Общее количество событий в истории: {monthly_totals['event_count'].sum():.0f}")
    print(f"Среднее количество событий в месяц: {monthly_totals['event_count'].mean():.2f}")
    print(f"Максимальное количество в месяц: {monthly_totals['event_count'].max():.0f}")
    
    if target_month:
        print(f"\nНа {target_month}:")
        print(f"Предсказано событий: {total_predicted:.0f}")
        print(f"Фактических событий: {total_actual:.0f}")
        print(f"Ошибка предсказания: {total_predicted - total_actual:+d}")
    
    return monthly_totals

def analyze_business_process(data, model_data=None, predictor=None, train_end_date=None, test_date=None,
                             bp_number=None, months_to_show=12, save_plots=False, predictions_df=None):
    """
    Основная функция анализа бизнес-процессов.
    Если передан predictions_df, используются готовые предсказания,
    иначе вызывается predict_for_month для генерации новых.
    """
    print("\n" + "="*80)
    print(f"АНАЛИЗ БИЗНЕС-ПРОЦЕССОВ")
    print(f"Обучающие данные до: {train_end_date}, Тестовая дата: {test_date}")
    if bp_number:
        print(f"Анализируемый бизнес-процесс: {bp_number}")
    else:
        print("Анализ всех бизнес-процессов")
    print("="*80)

    # 1. Разделяем данные на обучающую и тестовую части
    train_data, test_data = split_data_by_date(data, train_end_date, test_date)

    if len(test_data) == 0:
        print(f"Нет данных за {test_date} для анализа")
        return None

    # 2. Получаем предсказания
    if predictions_df is not None:
        print("Используются готовые предсказания из predictions_df")
        if 'prediction_month' in predictions_df.columns:
            predictions_df = predictions_df.rename(columns={'prediction_month': 'year_month'})
        if 'event_probability_count' in predictions_df.columns:
            predictions_df = predictions_df.rename(columns={'event_probability_count': 'predicted_probability'})

        required_cols = ['bp_number', 'risk_name', 'year_month', 'predicted_event_count']
        missing = [col for col in required_cols if col not in predictions_df.columns]
        if missing:
            print(f"Ошибка: в predictions_df отсутствуют колонки: {missing}")
            return None

        # Добавляем фактические значения и данные по источникам из test_data
        source_cols = ['source_1_count', 'source_2_count', 'source_3_count', 'source_4_count']
        test_data_subset = test_data[['bp_number', 'risk_name', 'event_count'] + source_cols].copy()

        predictions_df['bp_number'] = predictions_df['bp_number'].astype(str)
        test_data_subset['bp_number'] = test_data_subset['bp_number'].astype(str)
        predictions_df['risk_name'] = predictions_df['risk_name'].astype(str)
        test_data_subset['risk_name'] = test_data_subset['risk_name'].astype(str)

        predictions_df = predictions_df.merge(test_data_subset, on=['bp_number', 'risk_name'], how='left')

        if 'event_count' not in predictions_df.columns:
            print("Ошибка: после слияния не удалось получить фактические значения")
            return None

        predictions_df = predictions_df.rename(columns={'event_count': 'actual_event_count'})
        predictions_df['actual_event_count'] = predictions_df['actual_event_count'].fillna(0).astype(int)
        # Заполняем пропуски в source-колонках (если нет данных)
        for col in source_cols:
            if col in predictions_df.columns:
                predictions_df[col] = predictions_df[col].fillna(0).astype(int)

        predictions_df['prediction_error'] = predictions_df['predicted_event_count'] - predictions_df['actual_event_count']
        predictions_df['absolute_error'] = np.abs(predictions_df['prediction_error'])
        predictions_df['has_predicted_event'] = predictions_df['predicted_event_count'] > 0
        predictions_df['has_actual_event'] = predictions_df['actual_event_count'] > 0
        predictions_df['prediction_correct'] = predictions_df['has_predicted_event'] == predictions_df['has_actual_event']

        if 'predicted_probability' not in predictions_df.columns:
            if 'classifier_pred' in predictions_df.columns:
                predictions_df['predicted_probability'] = predictions_df['classifier_pred'].astype(float)
            else:
                predictions_df['predicted_probability'] = 0.5

        smape_val = improved_smape(predictions_df['actual_event_count'].values,
                                   predictions_df['predicted_event_count'].values)
        mae = predictions_df['absolute_error'].mean()
        accuracy = predictions_df['prediction_correct'].mean()

        tp = ((predictions_df['has_predicted_event']) & (predictions_df['has_actual_event'])).sum()
        fp = ((predictions_df['has_predicted_event']) & (~predictions_df['has_actual_event'])).sum()
        fn = ((~predictions_df['has_predicted_event']) & (predictions_df['has_actual_event'])).sum()
        tn = ((~predictions_df['has_predicted_event']) & (~predictions_df['has_actual_event'])).sum()

        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1 = 2 * precision * recall / (precision + recall + 1e-10) if (precision + recall) > 0 else 0
        rmse = np.sqrt(np.mean((predictions_df['predicted_event_count'] - predictions_df['actual_event_count'])**2))

        print(f"\n{'='*60}")
        print("МЕТРИКИ КАЧЕСТВА МОДЕЛИ (на основе готовых предсказаний):")
        print(f"{'='*60}")
        print(f"SMAPE: {smape_val:.1f}%")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1-Score: {f1:.3f}")
        print(f"\nМатрица ошибок: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
        print(f"\nПредсказано событий: {predictions_df['predicted_event_count'].sum():.0f}")
        print(f"Фактических событий: {predictions_df['actual_event_count'].sum():.0f}")
        print(f"Общее количество комбинаций: {len(predictions_df)}")
        print(f"{'='*60}")

    else:
        predictions_df = predict_for_month(
            train_data=train_data,
            test_data=test_data,
            model_data=model_data,
            predictor=predictor,
            target_month=test_date
        )

    if predictions_df is None or len(predictions_df) == 0:
        print("Не удалось создать предсказания")
        return None

    # 3. Если указан конкретный бизнес-процесс
    if bp_number is not None:
        bp_number_str = str(bp_number)
        available_bp_numbers = [str(bp) for bp in predictions_df['bp_number'].unique()]
        
        if bp_number_str not in available_bp_numbers:
            print(f"Бизнес-процесс {bp_number} не найден в предсказаниях")
            print(f"Доступные бизнес-процессы: {sorted(predictions_df['bp_number'].unique())}")
            return None
        
        print(f"\n{'='*80}")
        print(f"АНАЛИЗ БИЗНЕС-ПРОЦЕССА {bp_number}")
        print(f"{'='*80}")
        
        print("\n1. График сравнения предсказанных и фактических событий по рискам:")
        bp_predictions = visualize_bp_risks_comparison(predictions_df, bp_number)
        
        print("\n2. Временные ряды по каждому риску бизнес-процесса:")
        bp_mask = predictions_df['bp_number'].astype(str) == str(bp_number)
        bp_risks = predictions_df[bp_mask]['risk_name'].unique()
        
        for i, risk_name in enumerate(bp_risks, 1):
            risk_name_str = str(risk_name)
            print(f"\n   Риск {i}/{len(bp_risks)}: {risk_name_str}")
            risk_series = visualize_risk_time_series(
                data=train_data,
                predictions_df=predictions_df,
                bp_number=bp_number,
                risk_name=risk_name,
                months_to_show=months_to_show
            )
            
            if save_plots and risk_series is not None:
                safe_risk_name = str(risk_name).replace(' ', '_').replace('/', '_').replace('\\', '_')
                plt.savefig(f"bp_{bp_number}_risk_{safe_risk_name}_time_series.png", 
                          dpi=150, bbox_inches='tight')
                plt.close()
        
        print("\n3. Суммарный временной ряд по бизнес-процессу:")
        monthly_totals = visualize_bp_time_series(
            data=train_data,
            predictions_df=predictions_df,
            bp_number=bp_number,
            months_to_show=max(24, months_to_show)
        )
        
        if save_plots and monthly_totals is not None:
            plt.savefig(f"bp_{bp_number}_time_series.png", dpi=150, bbox_inches='tight')
            plt.close()
        
        return {
            'predictions': predictions_df,
            'bp_predictions': bp_predictions,
            'monthly_totals': monthly_totals
        }
    
    # 4. Если бизнес-процесс не указан - анализируем все
    else:
        print(f"\n{'='*80}")
        print(f"АНАЛИЗ ВСЕХ БИЗНЕС-ПРОЦЕССОВ")
        print(f"{'='*80}")
        
        all_bp_numbers = sorted(predictions_df['bp_number'].unique())
        print(f"Всего бизнес-процессов: {len(all_bp_numbers)}")
        
        results = {}
        
        for i, current_bp in enumerate(all_bp_numbers, 1):
            print(f"\n{'='*60}")
            print(f"Бизнес-процесс {i}/{len(all_bp_numbers)}: {current_bp}")
            print(f"{'='*60}")
            
            try:
                bp_predictions = visualize_bp_risks_comparison(predictions_df, current_bp)
                
                bp_risks = predictions_df[predictions_df['bp_number'].astype(str) == str(current_bp)]['risk_name'].unique()
                print(f"\nВременные ряды по всем рискам бизнес-процесса {current_bp} ({len(bp_risks)} рисков):")
                
                risks_to_display = bp_risks[:10] if len(bp_risks) > 10 else bp_risks
                
                for j, risk_name in enumerate(risks_to_display, 1):
                    risk_name_str = str(risk_name)
                    print(f"  Риск {j}/{len(risks_to_display)}: {risk_name_str}")
                    risk_series = visualize_risk_time_series(
                        data=train_data,
                        predictions_df=predictions_df,
                        bp_number=current_bp,
                        risk_name=risk_name,
                        months_to_show=months_to_show
                    )
                    
                    if save_plots and risk_series is not None:
                        safe_risk_name = str(risk_name).replace(' ', '_').replace('/', '_').replace('\\', '_')
                        plt.savefig(f"all_bp_{current_bp}_risk_{safe_risk_name}_time_series.png", 
                                  dpi=150, bbox_inches='tight')
                        plt.close()
                
                if len(bp_risks) > 10 and save_plots:
                    for risk_name in bp_risks[10:]:
                        try:
                            risk_name_str = str(risk_name)
                            risk_series = visualize_risk_time_series(
                                data=train_data,
                                predictions_df=predictions_df,
                                bp_number=current_bp,
                                risk_name=risk_name,
                                months_to_show=months_to_show
                            )
                            safe_risk_name = str(risk_name).replace(' ', '_').replace('/', '_').replace('\\', '_')
                            plt.savefig(f"all_bp_{current_bp}_risk_{safe_risk_name}_time_series.png", 
                                      dpi=150, bbox_inches='tight')
                            plt.close()
                        except Exception as e:
                            print(f"    Ошибка при сохранении графика для риска {risk_name_str}: {e}")
                            continue
                
                monthly_totals = visualize_bp_time_series(
                    data=train_data,
                    predictions_df=predictions_df,
                    bp_number=current_bp,
                    months_to_show=max(24, months_to_show)
                )
                
                if save_plots:
                    if bp_predictions is not None:
                        plt.figure(1)
                        plt.savefig(f"all_bp_{current_bp}_comparison.png", 
                                  dpi=150, bbox_inches='tight')
                        plt.close()
                    
                    if monthly_totals is not None:
                        plt.figure(2)
                        plt.savefig(f"all_bp_{current_bp}_time_series.png", 
                                  dpi=150, bbox_inches='tight')
                        plt.close()
                
                results[current_bp] = {
                    'predictions': bp_predictions,
                    'monthly_totals': monthly_totals,
                    'total_risks': len(bp_risks)
                }
                
                plt.close('all')
                
            except Exception as e:
                print(f"Ошибка при анализе бизнес-процесса {current_bp}: {e}")
                continue
        
        print(f"\n{'='*80}")
        print(f"СВОДНАЯ СТАТИСТИКА ПО ВСЕМ БИЗНЕС-ПРОЦЕССАМ")
        print(f"{'='*80}")
        
        total_predicted = predictions_df['predicted_event_count'].sum()
        total_actual = predictions_df['actual_event_count'].sum()
        total_error = total_predicted - total_actual
        
        bp_summary = predictions_df.groupby('bp_number').agg({
            'predicted_event_count': 'sum',
            'actual_event_count': 'sum',
            'risk_name': 'count'
        }).rename(columns={
            'predicted_event_count': 'total_predicted',
            'actual_event_count': 'total_actual',
            'risk_name': 'num_risks'
        }).reset_index()
        
        bp_summary['error'] = bp_summary['total_predicted'] - bp_summary['total_actual']
        bp_summary['absolute_error'] = abs(bp_summary['error'])
        
        print(f"\nОбщая статистика:")
        print(f"Всего бизнес-процессов: {len(bp_summary)}")
        print(f"Всего предсказано событий: {total_predicted:.0f}")
        print(f"Всего фактических событий: {total_actual:.0f}")
        print(f"Общая ошибка: {total_error:+.0f}")
        print(f"Средняя ошибка на бизнес-процесс: {bp_summary['error'].mean():.2f}")
        
        print(f"\nТоп-10 бизнес-процессов по предсказанным событиям:")
        top_bp = bp_summary.nlargest(10, 'total_predicted')
        for idx, row in top_bp.iterrows():
            print(f"  {row['bp_number']}: предсказано {row['total_predicted']:.0f}, "
                  f"факт {row['total_actual']:.0f}, ошибка {row['error']:+d} ({row['num_risks']} рисков)")
        
        return {
            'all_predictions': predictions_df,
            'bp_summary': bp_summary,
            'detailed_results': results
        }


# ================== ПРИМЕР ИСПОЛЬЗОВАНИЯ ==================
if __name__ == "__main__":
    # Загрузите свои данные
    # base_df = pd.read_csv('your_historical_data.csv')
    # result = pd.read_csv('your_predictions.csv')

    # Пример вызова для анализа всех бизнес-процессов с готовыми предсказаниями
    all_results = analyze_business_process(
        data=base_df,
        train_end_date="2025-08",
        test_date="2025-09",
        bp_number=None,
        months_to_show=12,
        save_plots=False,
        predictions_df=result
    )
    # pass